# Triple Barrier V/H/M Grid Screening and Internal Validation

**Load the Feature-Engineered Dataset**

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np
from itertools import product

PROJECT_DIR = Path(r"C:\UTAR - Master\Master Project\Project\Main Project")
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
RESULTS_DIR = PROJECT_DIR / "results"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(
    PROCESSED_DIR / "lseg_feature_engineered.csv",
    parse_dates=["Date"]
)

df = df.sort_values(
    ["Ticker", "Date"]
).reset_index(drop=True)

assert len(df) == 37798
assert df["Ticker"].nunique() == 13
assert not df.duplicated(["Ticker", "Date"]).any()

print("Total observations:", len(df))
print("Number of banks:", df["Ticker"].nunique())
print("Earliest date:", df["Date"].min())
print("Latest date:", df["Date"].max())


Total observations: 37798
Number of banks: 13
Earliest date: 2015-01-02 00:00:00
Latest date: 2026-10-30 00:00:00


**Calculate Historical Volatility for All Windows**

In [2]:
# Volatility windows to test
volatility_windows = list(range(2, 11))

# Holding periods and multipliers for later cells
holding_periods = list(range(1, 11))
multipliers = np.round(np.arange(0.5, 5.01, 0.5), 2).tolist()

# Reuse existing volatility_5 and volatility_10 from Notebook 02
# Calculate only volatility windows that are not yet available

new_windows = []

for window in volatility_windows:
    col = f"volatility_{window}"

    if col not in df.columns:
        df[col] = (
            df.groupby(["Ticker", "Segment_ID"])["returns"]
            .transform(
                lambda s: s.rolling(
                    window=window,
                    min_periods=window
                ).std(ddof=1)
            )
        )

        new_windows.append(window)

# Display availability
print("=== VOLATILITY WINDOWS ===")

for window in volatility_windows:
    col = f"volatility_{window}"

    print(
        f"V{window}:",
        int(df[col].notna().sum()),
        "available observations"
    )

print("\nNewly calculated windows:", new_windows)
print("Existing V5 and V10 reused from Notebook 02.")

# Display experimental search space
print("\n=== EXPERIMENTAL SEARCH SPACE ===")
print("Volatility windows:", volatility_windows)
print("Holding periods:", holding_periods)
print("Multipliers:", multipliers)

total = (
    len(volatility_windows)
    * len(holding_periods)
    * len(multipliers)
)

print("\nTotal configurations:", total)



=== VOLATILITY WINDOWS ===
V2: 37736 available observations
V3: 37705 available observations
V4: 37674 available observations
V5: 37643 available observations
V6: 37612 available observations
V7: 37581 available observations
V8: 37550 available observations
V9: 37519 available observations
V10: 37488 available observations

Newly calculated windows: [2, 3, 4, 6, 7, 8, 9]
Existing V5 and V10 reused from Notebook 02.

=== EXPERIMENTAL SEARCH SPACE ===
Volatility windows: [2, 3, 4, 5, 6, 7, 8, 9, 10]
Holding periods: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Multipliers: [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0]

Total configurations: 900


**Triple Barrier labelling function**

In [3]:
def generate_triple_barrier_labels(data, vol_window, horizon, multiplier):

    records = []
    volatility_column = f"volatility_{vol_window}"

    for (ticker, segment_id), stock in data.groupby(
        ["Ticker", "Segment_ID"],
        sort=False
    ):
        stock = stock.sort_values("Date").reset_index(drop=True)

        close = stock["Close"].to_numpy(dtype=float)
        volatility = stock[volatility_column].to_numpy(dtype=float)
        dates = stock["Date"].to_numpy()

        for i in range(len(stock) - horizon):

            sigma = volatility[i]

            if not np.isfinite(sigma) or sigma <= 0:
                continue

            entry_price = close[i]

            upper_barrier = entry_price * (1 + multiplier * sigma)
            lower_barrier = entry_price * (1 - multiplier * sigma)

            future_prices = close[i + 1:i + horizon + 1]

            upper_hits = np.flatnonzero(
                future_prices >= upper_barrier
            )

            lower_hits = np.flatnonzero(
                future_prices <= lower_barrier
            )

            first_upper = (
                int(upper_hits[0]) if len(upper_hits) else horizon
            )

            first_lower = (
                int(lower_hits[0]) if len(lower_hits) else horizon
            )

            if first_upper < first_lower:
                label = 1
                event_position = first_upper
                barrier_type = "Upper"

            elif first_lower < first_upper:
                label = -1
                event_position = first_lower
                barrier_type = "Lower"

            else:
                label = 0
                event_position = horizon - 1
                barrier_type = "Time"

            records.append({
                "Ticker": ticker,
                "Country": stock.loc[i, "Country"],
                "Date": dates[i],
                "Event_End_Date": dates[i + event_position + 1],
                "Label": label,
                "Barrier_Type": barrier_type,
                "Holding_Days": event_position + 1,
                "Volatility_Window": vol_window,
                "Holding_Period": horizon,
                "Multiplier": multiplier
            })

    return pd.DataFrame(records)


# Test the function using our original configuration
baseline_labels = generate_triple_barrier_labels(
    df,
    vol_window=5,
    horizon=5,
    multiplier=2.0
)

# Validate labels
assert baseline_labels["Label"].isin([-1, 0, 1]).all()
assert (baseline_labels["Event_End_Date"] > baseline_labels["Date"]).all()
assert baseline_labels["Holding_Days"].between(1, 5).all()
assert not baseline_labels.duplicated(["Ticker", "Date"]).any()

# Check original V5_H5_M2 class distribution
baseline_validation = baseline_labels.loc[
    (baseline_labels["Date"] >= "2023-01-01")
    & (baseline_labels["Date"] <= "2024-12-31")
    & (baseline_labels["Event_End_Date"] <= "2024-12-31")
]

baseline_distribution = (
    baseline_validation
    .groupby(["Country", "Label"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=[-1, 0, 1], fill_value=0)
)

baseline_distribution = (
    baseline_distribution
    .div(baseline_distribution.sum(axis=1), axis=0)
    .mul(100)
    .round(2)
)

baseline_distribution.columns = ["Down_Pct", "Flat_Pct", "Up_Pct"]

print("=== ORIGINAL V5_H5_M2 CLASS DISTRIBUTION (2023–2024) ===")
display(baseline_distribution)

print("\n=== SAMPLE LABELLED OBSERVATIONS ===")
display(baseline_labels.head(10))


=== ORIGINAL V5_H5_M2 CLASS DISTRIBUTION (2023–2024) ===


,Down_Pct,Flat_Pct,Up_Pct
Country,,,
Malaysia,23.94,48.05,28.01
Singapore,25.47,44.22,30.31



=== SAMPLE LABELLED OBSERVATIONS ===


,Ticker,Country,Date,Event_End_Date,Label,Barrier_Type,Holding_Days,Volatility_Window,Holding_Period,Multiplier
0,1015.KL,Malaysia,2015-01-09,2015-01-16,0,Time,5,5,5,2.0
1,1015.KL,Malaysia,2015-01-12,2015-01-19,0,Time,5,5,5,2.0
2,1015.KL,Malaysia,2015-01-13,2015-01-20,0,Time,5,5,5,2.0
3,1015.KL,Malaysia,2015-01-14,2015-01-21,0,Time,5,5,5,2.0
4,1015.KL,Malaysia,2015-01-15,2015-01-20,1,Upper,3,5,5,2.0
5,1015.KL,Malaysia,2015-01-16,2015-01-20,1,Upper,2,5,5,2.0
6,1015.KL,Malaysia,2015-01-19,2015-01-21,1,Upper,2,5,5,2.0
7,1015.KL,Malaysia,2015-01-20,2015-01-22,1,Upper,2,5,5,2.0
8,1015.KL,Malaysia,2015-01-21,2015-01-22,1,Upper,1,5,5,2.0
9,1015.KL,Malaysia,2015-01-22,2015-01-29,-1,Lower,5,5,5,2.0


**Prepare Data for Parameter Comparison**

In [4]:

# Prepare the same observation sample for all V/H/M comparisons

vol_cols = [f"volatility_{v}" for v in volatility_windows]

groups = df.groupby(["Ticker", "Segment_ID"], sort=False)

# Obtain the next 10 closing prices within each continuous segment
future_cols = []

for h in holding_periods:
    col = f"future_close_{h}"
    df[col] = groups["Close"].shift(-h)
    future_cols.append(col)

# Date of the 10th future trading session
df["future_date_10"] = groups["Date"].shift(-10)

# Require valid historical volatility for every candidate window
valid_volatility = (
    np.isfinite(df[vol_cols].to_numpy(dtype=float)).all(axis=1)
    & (df[vol_cols] > 0).all(axis=1)
)

# Require all 10 future closing prices
valid_future = df[future_cols].notna().all(axis=1)

# Restrict parameter screening to historical development data
screening_mask = (
    (df["Date"] >= "2015-01-01")
    & (df["Date"] <= "2022-12-31")
    & (df["future_date_10"] <= "2022-12-31")
    & valid_volatility
    & valid_future
)

screening_df = df.loc[
    screening_mask,
    ["Ticker", "Country", "Date"] + vol_cols + future_cols + ["Close"]
].copy()

print("=== PARAMETER SCREENING SAMPLE ===")
print("Total observations:", len(screening_df))
print("Banks:", screening_df["Ticker"].nunique())
print("Date range:", screening_df["Date"].min(),
      "to", screening_df["Date"].max())

print("\nObservations by market:")
display(screening_df.groupby("Country").size())

print("\nConfigurations to compare:",
      len(volatility_windows) * len(holding_periods) * len(multipliers))


=== PARAMETER SCREENING SAMPLE ===
Total observations: 24295
Banks: 13
Date range: 2015-01-16 00:00:00 to 2022-12-15 00:00:00

Observations by market:


Country
Malaysia     18344
Singapore     5951
dtype: int64


Configurations to compare: 900


**Test All 900 Triple Barrier Configurations**

In [5]:

# Prepare price arrays
current_prices = screening_df["Close"].to_numpy(dtype=float)

future_prices = screening_df[
    [f"future_close_{h}" for h in holding_periods]
].to_numpy(dtype=float)

market_indices = {
    country: np.where(screening_df["Country"].to_numpy() == country)[0]
    for country in ["Malaysia", "Singapore"]
}

results = []
baseline_check = None

for v in volatility_windows:

    volatility = screening_df[
        f"volatility_{v}"
    ].to_numpy(dtype=float)

    for h in holding_periods:

        future = future_prices[:, :h]

        for m in multipliers:

            upper = current_prices * (1 + m * volatility)
            lower = current_prices * (1 - m * volatility)

            upper_hits = future >= upper[:, None]
            lower_hits = future <= lower[:, None]

            first_upper = np.where(
                upper_hits.any(axis=1),
                upper_hits.argmax(axis=1) + 1,
                h + 1
            )

            first_lower = np.where(
                lower_hits.any(axis=1),
                lower_hits.argmax(axis=1) + 1,
                h + 1
            )

            labels = np.where(
                first_upper < first_lower, 1,
                np.where(first_lower < first_upper, -1, 0)
            )

            if v == 5 and h == 5 and m == 2.0:
                baseline_check = labels.copy()

            for country, idx in market_indices.items():

                counts = np.bincount(
                    labels[idx] + 1,
                    minlength=3
                )

                total = counts.sum()

                results.append({
                    "Country": country,
                    "V": v,
                    "H": h,
                    "M": m,
                    "Total": int(total),
                    "Down_Pct": counts[0] / total * 100,
                    "Flat_Pct": counts[1] / total * 100,
                    "Up_Pct": counts[2] / total * 100
                })

parameter_summary = pd.DataFrame(results)

# Verify original labelling logic on the common sample
original_labels = (
    baseline_labels
    .set_index(["Ticker", "Date"])["Label"]
    .reindex(
        pd.MultiIndex.from_frame(
            screening_df[["Ticker", "Date"]]
        )
    )
)

assert original_labels.notna().all()
assert np.array_equal(
    baseline_check,
    original_labels.to_numpy(dtype=int)
)

assert len(parameter_summary) == 1800

# Display original configuration on 2015–2022 sample
print("=== V5_H5_M2: HISTORICAL SCREENING SAMPLE ===")

display(
    parameter_summary.loc[
        (parameter_summary["V"] == 5)
        & (parameter_summary["H"] == 5)
        & (parameter_summary["M"] == 2.0)
    ].round(2)
)

# Examine multiplier effect while keeping V5 and H5
print("\n=== EFFECT OF MULTIPLIER: V5_H5 ===")

display(
    parameter_summary.loc[
        (parameter_summary["V"] == 5)
        & (parameter_summary["H"] == 5)
    ].pivot(
        index="M",
        columns="Country",
        values="Flat_Pct"
    ).round(2)
)

# Count configurations meeting the supervisor's Flat target
flat_comparison = parameter_summary.pivot(
    index=["V", "H", "M"],
    columns="Country",
    values="Flat_Pct"
)

both_target = (
    flat_comparison["Malaysia"].between(30, 35)
    & flat_comparison["Singapore"].between(30, 35)
)

print("\nConfigurations with 30–35% Flat in BOTH markets:",
      int(both_target.sum()))


=== V5_H5_M2: HISTORICAL SCREENING SAMPLE ===


,Country,V,H,M,Total,Down_Pct,Flat_Pct,Up_Pct
686,Malaysia,5,5,2.0,18344,26.80,47.29,25.92
687,Singapore,5,5,2.0,5951,27.96,42.01,30.03



=== EFFECT OF MULTIPLIER: V5_H5 ===


Country,Malaysia,Singapore
M,,
0.5,4.52,1.85
1.0,18.29,12.67
1.5,33.98,28.43
2.0,47.29,42.01
2.5,57.89,52.98
3.0,66.17,62.33
3.5,72.33,69.58
4.0,77.45,75.43
4.5,81.38,79.67



Configurations with 30–35% Flat in BOTH markets: 0


**Compare All Three Class Percentages**

In [6]:

# Compare class balance across all 900 configurations

target_pct = 100 / 3

balance_results = parameter_summary.copy()

# Measure the largest deviation from an equal three-class distribution
class_cols = ["Down_Pct", "Flat_Pct", "Up_Pct"]

balance_results["Balance_Gap"] = (
    balance_results[class_cols]
    .sub(target_pct)
    .abs()
    .max(axis=1)
)

# Best configurations for each market
for country in ["Malaysia", "Singapore"]:

    print(f"\n=== {country.upper()}: TOP 15 MOST BALANCED CONFIGURATIONS ===")

    country_results = (
        balance_results[
            balance_results["Country"] == country
        ]
        .sort_values("Balance_Gap")
    )

    display(
        country_results[
            ["V", "H", "M",
             "Down_Pct", "Flat_Pct", "Up_Pct",
             "Balance_Gap"]
        ].head(15).round(2)
    )

# Compare balance across BOTH markets
combined = balance_results.pivot(
    index=["V", "H", "M"],
    columns="Country",
    values=[
        "Down_Pct", "Flat_Pct",
        "Up_Pct", "Balance_Gap"
    ]
)

combined.columns = [
    f"{country}_{metric}"
    for metric, country in combined.columns
]

combined = combined.reset_index()

# Use the worse market's imbalance as the overall score
combined["Worst_Balance_Gap"] = combined[
    [
        "Malaysia_Balance_Gap",
        "Singapore_Balance_Gap"
    ]
].max(axis=1)

combined = combined.sort_values("Worst_Balance_Gap")

print("\n=== TOP 20 MOST BALANCED CONFIGURATIONS: BOTH MARKETS ===")

display(
    combined[
        [
            "V", "H", "M",
            "Malaysia_Down_Pct",
            "Malaysia_Flat_Pct",
            "Malaysia_Up_Pct",
            "Singapore_Down_Pct",
            "Singapore_Flat_Pct",
            "Singapore_Up_Pct",
            "Worst_Balance_Gap"
        ]
    ].head(20).round(2)
)



=== MALAYSIA: TOP 15 MOST BALANCED CONFIGURATIONS ===


,V,H,M,Down_Pct,Flat_Pct,Up_Pct,Balance_Gap
148,2,8,2.5,33.90,33.35,32.75,0.59
946,6,8,2.0,33.72,33.80,32.48,0.85
842,6,3,1.0,34.39,33.09,32.52,1.06
746,5,8,2.0,34.41,33.06,32.53,1.08
1766,10,9,2.0,34.42,32.62,32.96,1.09
684,5,5,1.5,33.81,33.98,32.21,1.12
388,3,10,2.5,34.49,32.89,32.62,1.16
1042,7,3,1.0,34.03,33.81,32.16,1.18
326,3,7,2.0,33.85,34.04,32.10,1.23
64,2,4,1.5,33.62,34.30,32.08,1.25



=== SINGAPORE: TOP 15 MOST BALANCED CONFIGURATIONS ===


,V,H,M,Down_Pct,Flat_Pct,Up_Pct,Balance_Gap
465,4,4,1.5,32.01,33.78,34.21,1.32
45,2,3,1.5,31.93,33.54,34.53,1.41
201,3,1,0.5,31.86,34.16,33.98,1.47
507,4,6,2.0,31.84,34.08,34.08,1.49
349,3,8,2.5,31.86,32.94,35.20,1.87
769,5,9,2.5,31.32,33.39,35.29,2.01
969,6,9,2.5,31.31,33.93,34.77,2.03
1327,8,7,2.0,31.26,33.32,35.42,2.09
109,2,6,2.5,31.20,34.33,34.46,2.13
151,2,8,3.0,31.19,34.15,34.67,2.15



=== TOP 20 MOST BALANCED CONFIGURATIONS: BOTH MARKETS ===


,V,H,M,Malaysia_Down_Pct,Malaysia_Flat_Pct,Malaysia_Up_Pct,Singapore_Down_Pct,Singapore_Flat_Pct,Singapore_Up_Pct,Worst_Balance_Gap
11,2,2,1.0,32.39,36.42,31.19,33.12,31.14,35.74,3.09
64,2,7,2.5,32.44,36.33,31.23,32.92,30.62,36.46,3.13
43,2,5,2.0,32.34,36.58,31.08,33.20,30.58,36.21,3.25
642,8,5,1.5,32.53,36.55,30.93,32.92,30.40,36.68,3.35
542,7,5,1.5,32.78,35.92,31.30,32.89,30.38,36.73,3.40
363,5,7,2.0,32.38,36.83,30.78,32.97,30.80,36.23,3.50
132,3,4,1.5,32.39,36.88,30.73,32.92,31.44,35.64,3.55
394,5,10,2.5,32.30,36.60,31.09,32.92,30.01,37.07,3.74
742,9,5,1.5,32.33,37.09,30.58,32.89,30.65,36.46,3.75
873,10,8,2.0,32.41,36.46,31.13,32.62,30.28,37.10,3.77


**Prepare the Three Approaches**


| Approach | MY configuration | SG configuration |
| :--- | :--- | :--- |
| A — Original baseline | V5_H5_M2 | V5_H5_M2 |
| B — Best common class balance | V2_H2_M1 | V2_H2_M1 |
| C — Best market-specific class balance | V2_H8_M2.5 | V4_H4_M1.5 |


In [7]:

# Define three experimental approaches
candidate_configs = {
    "A_Original": {
        "Malaysia": (5, 5, 2.0),
        "Singapore": (5, 5, 2.0)
    },
    "B_Common": {
        "Malaysia": (2, 2, 1.0),
        "Singapore": (2, 2, 1.0)
    },
    "C_Market_Specific": {
        "Malaysia": (2, 8, 2.5),
        "Singapore": (4, 4, 1.5)
    }
}

# Reuse the original 23 model features from Notebook 04
feature_columns = [
    "returns", "returns_lag_1", "returns_lag_2",
    "momentum_5", "momentum_10",
    "daily_range", "candle_body",
    "sma_5_ratio", "sma_10_ratio", "sma_20_ratio",
    "volatility_5", "volatility_10", "volatility_20",
    "rsi", "macd", "macd_signal", "macd_diff",
    "volume_change", "volume_ratio_20",
    "bb_high", "bb_low", "bb_width", "skewness_20"
]

# Use the common observation sample from Cell 4
common_data = screening_df[
    ["Ticker", "Country", "Date"]
].merge(
    df[
        ["Ticker", "Date", "future_date_10"] + feature_columns
    ],
    on=["Ticker", "Date"],
    how="left",
    validate="one_to_one"
)

common_data = common_data.dropna(
    subset=feature_columns
).copy()

# Ensure all training horizons end within 2020
train_mask = (
    (common_data["Date"] < "2021-01-01")
    & (common_data["future_date_10"] <= "2020-12-31")
)

validation_mask = common_data["Date"].between(
    "2021-01-01", "2022-12-31"
)

common_data = common_data.loc[
    train_mask | validation_mask
].copy()

common_data["Split"] = np.where(
    common_data["Date"] < "2021-01-01",
    "Train",
    "Validation"
)

# Generate labels using the same function as Notebook 03
label_cache = {}
candidate_records = []

for approach, markets in candidate_configs.items():

    for country, params in markets.items():

        if params not in label_cache:
            label_cache[params] = generate_triple_barrier_labels(
                df, *params
            )

        labels = label_cache[params][
            ["Ticker", "Date", "Event_End_Date", "Label"]
        ]

        market_data = common_data.loc[
            common_data["Country"] == country
        ].merge(
            labels,
            on=["Ticker", "Date"],
            how="inner",
            validate="one_to_one"
        )

        market_data["Approach"] = approach
        market_data["V"] = params[0]
        market_data["H"] = params[1]
        market_data["M"] = params[2]

        candidate_records.append(market_data)

candidate_data = pd.concat(
    candidate_records,
    ignore_index=True
)

# Verify all labels mature within their respective periods
period_end = np.where(
    candidate_data["Split"] == "Train",
    np.datetime64("2020-12-31"),
    np.datetime64("2022-12-31")
)

assert (
    candidate_data["Event_End_Date"].to_numpy()
    <= period_end
).all()

assert candidate_data["Label"].isin([-1, 0, 1]).all()

print("=== TRAINING AND VALIDATION OBSERVATIONS ===")

display(
    candidate_data.groupby(
        ["Approach", "Country", "Split"]
    ).size().unstack(fill_value=0)
)

print("\n=== VALIDATION CLASS DISTRIBUTIONS (2021–2022) ===")

validation_distribution = (
    candidate_data[
        candidate_data["Split"] == "Validation"
    ]
    .groupby(["Approach", "Country", "Label"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=[-1, 0, 1], fill_value=0)
)

validation_distribution = (
    validation_distribution
    .div(validation_distribution.sum(axis=1), axis=0)
    .mul(100)
    .round(2)
)

validation_distribution.columns = [
    "Down_Pct", "Flat_Pct", "Up_Pct"
]

display(validation_distribution)


=== TRAINING AND VALIDATION OBSERVATIONS ===


Split                        Train  Validation
Approach          Country                     
A_Original        Malaysia   13192        4493
                  Singapore   4373        1479
B_Common          Malaysia   13192        4493
                  Singapore   4373        1479
C_Market_Specific Malaysia   13192        4493
                  Singapore   4373        1479


=== VALIDATION CLASS DISTRIBUTIONS (2021–2022) ===


Down_Pct  Flat_Pct  Up_Pct
Approach          Country                              
A_Original        Malaysia      24.37     48.30   27.33
                  Singapore     25.90     45.57   28.53
B_Common          Malaysia      31.45     36.30   32.25
                  Singapore     30.70     32.66   36.65
C_Market_Specific Malaysia      31.58     34.05   34.36
                  Singapore     29.55     35.97   34.48

**Compare XGBoost Performance**

In [8]:

%pip install xgboost

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:

%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:

from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score
)

# Original XGBoost settings from Notebook 04
model_params = {
    "n_estimators": 200,
    "learning_rate": 0.05,
    "max_depth": 3,
    "min_child_weight": 3,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "objective": "multi:softprob",
    "num_class": 3,
    "eval_metric": "mlogloss",
    "tree_method": "hist",
    "random_state": 42,
    "n_jobs": -1
}

model_results = []
model_predictions = []

for approach in candidate_configs:

    for country in ["Malaysia", "Singapore"]:

        subset = candidate_data.loc[
            (candidate_data["Approach"] == approach)
            & (candidate_data["Country"] == country)
        ]

        train = subset[subset["Split"] == "Train"]
        valid = subset[subset["Split"] == "Validation"]

        X_train = train[feature_columns]
        y_train = train["Label"].to_numpy() + 1

        X_valid = valid[feature_columns]
        y_valid = valid["Label"].to_numpy() + 1

        assert set(np.unique(y_train)) == {0, 1, 2}
        assert X_train.notna().all().all()
        assert X_valid.notna().all().all()

        model = XGBClassifier(**model_params)

        model.fit(
            X_train,
            y_train,
            verbose=False
        )

        y_pred = model.predict(X_valid)

        # Majority-class baseline based on training data
        majority_class = int(
            pd.Series(y_train).value_counts().idxmax()
        )

        baseline_pred = np.full(
            len(y_valid),
            majority_class
        )

        # Per-class recall and precision
        recalls = recall_score(
            y_valid, y_pred,
            labels=[0, 1, 2],
            average=None,
            zero_division=0
        )

        precisions = precision_score(
            y_valid, y_pred,
            labels=[0, 1, 2],
            average=None,
            zero_division=0
        )

        model_results.append({
            "Approach": approach,
            "Country": country,
            "Accuracy": accuracy_score(y_valid, y_pred),
            "Baseline_Accuracy": accuracy_score(
                y_valid, baseline_pred
            ),
            "Balanced_Accuracy": balanced_accuracy_score(
                y_valid, y_pred
            ),
            "Macro_F1": f1_score(
                y_valid, y_pred,
                average="macro"
            ),
            "MCC": matthews_corrcoef(y_valid, y_pred),
            "Down_Recall": recalls[0],
            "Flat_Recall": recalls[1],
            "Up_Recall": recalls[2],
            "Directional_Recall": (
                recalls[0] + recalls[2]
            ) / 2,
            "Down_Precision": precisions[0],
            "Up_Precision": precisions[2],
            "Predicted_Flat_Pct": (
                (y_pred == 1).mean() * 100
            )
        })

        pred_df = valid[
            ["Ticker", "Date", "Label"]
        ].copy()

        pred_df["Approach"] = approach
        pred_df["Country"] = country
        pred_df["Predicted_Label"] = y_pred - 1

        model_predictions.append(pred_df)

results_df = pd.DataFrame(model_results)

prediction_results = pd.concat(
    model_predictions,
    ignore_index=True
)

# Convert selected metrics to percentages
percentage_columns = [
    "Accuracy",
    "Baseline_Accuracy",
    "Balanced_Accuracy",
    "Down_Recall",
    "Flat_Recall",
    "Up_Recall",
    "Directional_Recall",
    "Down_Precision",
    "Up_Precision"
]

display_results = results_df.copy()

display_results[percentage_columns] *= 100

print("=== XGBOOST PERFORMANCE COMPARISON (2021–2022) ===")

display(
    display_results.round(4)
    .sort_values(["Country", "Approach"])
)


=== XGBOOST PERFORMANCE COMPARISON (2021–2022) ===


,Approach,Country,Accuracy,Baseline_Accuracy,Balanced_Accuracy,Macro_F1,MCC,Down_Recall,Flat_Recall,Up_Recall,Directional_Recall,Down_Precision,Up_Precision,Predicted_Flat_Pct
0,A_Original,Malaysia,54.0396,48.2974,45.1297,0.4361,0.2349,32.2374,85.8065,17.3453,24.7914,39.4855,42.1782,68.8627
2,B_Common,Malaysia,50.3672,36.3009,49.6771,0.4881,0.2573,53.3616,67.4433,28.2264,40.7940,42.2409,41.0231,38.0815
4,C_Market_Specific,Malaysia,52.2591,31.5825,52.4726,0.5145,0.2928,58.0691,71.0458,28.3031,43.1861,42.2781,42.3040,33.6301
1,A_Original,Singapore,48.4111,45.5713,43.3403,0.4303,0.1842,26.3708,70.4748,33.1754,29.7731,34.4710,33.4928,51.9270
3,B_Common,Singapore,46.7208,36.6464,46.4049,0.4678,0.1967,33.4802,57.7640,47.9705,40.7253,34.4671,42.6929,29.0061
5,C_Market_Specific,Singapore,44.4219,34.4828,43.6183,0.4341,0.1605,32.0366,59.2105,39.6078,35.8222,37.7358,38.3302,39.2833


**Examine the confusion matrices**

In [11]:

# Examine prediction behaviour for all three approaches

class_names = {-1: "Down", 0: "Flat", 1: "Up"}

for country in ["Malaysia", "Singapore"]:
    for approach in candidate_configs:

        subset = prediction_results.loc[
            (prediction_results["Country"] == country)
            & (prediction_results["Approach"] == approach)
        ]

        confusion = pd.crosstab(
            subset["Label"],
            subset["Predicted_Label"]
        ).reindex(
            index=[-1, 0, 1],
            columns=[-1, 0, 1],
            fill_value=0
        )

        recall_matrix = confusion.div(
            confusion.sum(axis=1).replace(0, np.nan),
            axis=0
        ) * 100

        recall_matrix = recall_matrix.rename(
            index=class_names,
            columns=class_names
        )

        predicted_pct = (
            subset["Predicted_Label"]
            .value_counts(normalize=True)
            .reindex([-1, 0, 1], fill_value=0)
            .mul(100)
        )

        print(f"\n=== {country.upper()} — {approach} ===")

        print("Confusion matrix (% of each actual class):")
        display(recall_matrix.round(2))

        print("Predicted class distribution (%):")
        display(
            pd.DataFrame({
                "Class": ["Down", "Flat", "Up"],
                "Predicted_Pct": predicted_pct.values
            }).round(2)
        )



=== MALAYSIA — A_Original ===
Confusion matrix (% of each actual class):


Predicted_Label,Down,Flat,Up
Label,,,
Down,32.24,51.96,15.80
Flat,8.71,85.81,5.48
Up,28.66,53.99,17.35


Predicted class distribution (%):


,Class,Predicted_Pct
0,Down,19.90
1,Flat,68.86
2,Up,11.24



=== MALAYSIA — B_Common ===
Confusion matrix (% of each actual class):


Predicted_Label,Down,Flat,Up
Label,,,
Down,53.36,20.81,25.83
Flat,18.88,67.44,13.67
Up,49.90,21.88,28.23


... (14 outputs left out)


=== SINGAPORE — C_Market_Specific ===
Confusion matrix (% of each actual class):


Predicted_Label,Down,Flat,Up
Label,,,
Down,32.04,22.88,45.08
Flat,16.73,59.21,24.06
Up,27.84,32.55,39.61


Predicted class distribution (%):


,Class,Predicted_Pct
0,Down,25.08
1,Flat,39.28
2,Up,35.63


**Bank-Level Performance Analysis**

In [12]:

bank_results = []

for (country, ticker, approach), group in prediction_results.groupby(
    ["Country", "Ticker", "Approach"]
):
    y_true = group["Label"].to_numpy()
    y_pred = group["Predicted_Label"].to_numpy()

    recalls = recall_score(
        y_true, y_pred,
        labels=[-1, 0, 1],
        average=None,
        zero_division=0
    )

    precisions = precision_score(
        y_true, y_pred,
        labels=[-1, 0, 1],
        average=None,
        zero_division=0
    )

    up_mask = y_true == 1
    down_mask = y_true == -1

    bank_results.append({
        "Country": country,
        "Ticker": ticker,
        "Approach": approach,
        "N": len(group),
        "Accuracy": accuracy_score(y_true, y_pred) * 100,
        "Down_Recall": recalls[0] * 100,
        "Flat_Recall": recalls[1] * 100,
        "Up_Recall": recalls[2] * 100,
        "Directional_Recall": (recalls[0] + recalls[2]) * 50,
        "Down_Precision": precisions[0] * 100,
        "Flat_Precision": precisions[1] * 100,
        "Up_Precision": precisions[2] * 100,
        "Up_Predicted_Down": (
            (y_pred[up_mask] == -1).mean() * 100
            if up_mask.any() else np.nan
        ),
        "Down_Predicted_Up": (
            (y_pred[down_mask] == 1).mean() * 100
            if down_mask.any() else np.nan
        ),
        "Actual_Down_N": int(down_mask.sum()),
        "Actual_Flat_N": int((y_true == 0).sum()),
        "Actual_Up_N": int(up_mask.sum())
    })

bank_results_df = pd.DataFrame(bank_results)

for country in ["Malaysia", "Singapore"]:

    subset = bank_results_df[
        bank_results_df["Country"] == country
    ].sort_values(["Ticker", "Approach"])

    print(f"\n=== {country.upper()}: BANK-LEVEL RECALL AND ACCURACY ===")

    display(
        subset[
            [
                "Ticker", "Approach", "N",
                "Accuracy",
                "Down_Recall", "Flat_Recall", "Up_Recall",
                "Directional_Recall"
            ]
        ].round(2)
    )

    print(f"\n=== {country.upper()}: PRECISION AND WRONG-DIRECTION ERRORS ===")

    display(
        subset[
            [
                "Ticker", "Approach",
                "Down_Precision", "Flat_Precision", "Up_Precision",
                "Up_Predicted_Down", "Down_Predicted_Up",
                "Actual_Down_N", "Actual_Flat_N", "Actual_Up_N"
            ]
        ].round(2)
    )



=== MALAYSIA: BANK-LEVEL RECALL AND ACCURACY ===


,Ticker,Approach,N,Accuracy,Down_Recall,Flat_Recall,Up_Recall,Directional_Recall
0,1015.KL,A_Original,424,54.95,24.44,88.99,14.66,19.55
1,1015.KL,B_Common,424,56.13,50.82,80.50,33.57,42.19
2,1015.KL,C_Market_Specific,424,55.19,46.15,79.75,34.72,40.44
3,1023.KL,A_Original,471,53.93,25.62,89.40,21.80,23.71
4,1023.KL,B_Common,471,51.59,51.01,71.24,34.32,42.66
5,1023.KL,C_Market_Specific,471,51.38,44.72,75.35,37.50,41.11
6,1066.KL,A_Original,472,49.58,38.52,80.37,8.40,23.46
7,1066.KL,B_Common,472,49.36,55.28,63.25,26.90,41.09
8,1066.KL,C_Market_Specific,472,55.72,68.18,71.85,28.57,48.38
9,1155.KL,A_Original,471,46.71,39.39,73.13,10.34,24.87



=== MALAYSIA: PRECISION AND WRONG-DIRECTION ERRORS ===


,Ticker,Approach,Down_Precision,Flat_Precision,Up_Precision,Up_Predicted_Down,Down_Predicted_Up,Actual_Down_N,Actual_Flat_N,Actual_Up_N
0,1015.KL,A_Original,34.92,60.25,43.59,24.14,12.22,90,218,116
1,1015.KL,B_Common,43.97,65.98,53.93,40.56,25.41,122,159,143
2,1015.KL,C_Market_Specific,39.13,74.29,45.05,46.53,38.46,117,163,144
3,1023.KL,A_Original,56.36,56.23,40.85,9.77,24.79,121,217,133
4,1023.KL,B_Common,46.06,58.60,48.33,40.83,25.50,149,153,169
5,1023.KL,C_Market_Specific,48.32,62.94,41.45,39.29,40.37,161,142,168
6,1066.KL,A_Original,42.34,56.05,23.40,32.06,12.30,122,219,131
7,1066.KL,B_Common,43.63,66.04,35.78,56.55,26.09,161,166,145
8,1066.KL,C_Market_Specific,50.21,74.62,44.66,62.73,21.02,176,135,161
9,1155.KL,A_Original,24.07,59.50,50.00,46.90,9.09,99,227,145



=== SINGAPORE: BANK-LEVEL RECALL AND ACCURACY ===


,Ticker,Approach,N,Accuracy,Down_Recall,Flat_Recall,Up_Recall,Directional_Recall
30,D05.SI,A_Original,493,48.48,22.58,75.46,31.37,26.98
31,D05.SI,B_Common,493,45.03,30.07,58.49,45.86,37.96
32,D05.SI,C_Market_Specific,493,46.86,31.51,67.44,39.43,35.47
33,O39.SI,A_Original,493,45.84,23.26,67.73,32.64,27.95
34,O39.SI,B_Common,493,48.48,37.75,58.94,48.69,43.22
35,O39.SI,C_Market_Specific,493,42.80,30.61,54.76,41.57,36.09
36,U11.SI,A_Original,493,50.91,33.08,68.49,36.00,34.54
37,U11.SI,B_Common,493,46.65,32.67,56.07,49.41,41.04
38,U11.SI,C_Market_Specific,493,43.61,34.03,55.73,37.58,35.80



=== SINGAPORE: PRECISION AND WRONG-DIRECTION ERRORS ===


,Ticker,Approach,Down_Precision,Flat_Precision,Up_Precision,Up_Predicted_Down,Down_Predicted_Up,Actual_Down_N,Actual_Flat_N,Actual_Up_N
30,D05.SI,A_Original,32.18,56.60,40.68,23.53,32.26,124,216,153
31,D05.SI,B_Common,32.62,62.00,41.09,38.12,51.63,153,159,181
32,D05.SI,C_Market_Specific,41.44,51.56,43.95,24.57,36.99,146,172,175
33,O39.SI,A_Original,29.13,61.83,31.54,27.78,49.61,129,220,144
34,O39.SI,B_Common,36.77,62.68,47.45,36.13,46.36,151,151,191
35,O39.SI,C_Market_Specific,33.09,53.18,40.22,30.90,47.62,147,168,178
36,U11.SI,A_Original,41.75,68.20,29.80,26.40,44.62,130,238,125
37,U11.SI,B_Common,33.79,70.80,39.81,35.88,57.33,150,173,170
38,U11.SI,C_Market_Specific,39.52,58.47,31.72,28.03,50.69,144,192,157


**Save Experimental Results**

In [13]:

output_dir = RESULTS_DIR / "notebook06"
output_dir.mkdir(parents=True, exist_ok=True)

parameter_summary.to_csv(
    output_dir / "tb_900_parameter_screening.csv",
    index=False
)

validation_distribution.to_csv(
    output_dir / "abc_validation_class_distributions.csv"
)

results_df.to_csv(
    output_dir / "abc_xgboost_performance.csv",
    index=False
)

prediction_results.to_csv(
    output_dir / "abc_predictions_2021_2022.csv",
    index=False
)

bank_results_df.to_csv(
    output_dir / "abc_bank_level_performance.csv",
    index=False
)

print("Saved files:")
for file in sorted(output_dir.glob("*.csv")):
    print(file.name)


Saved files:
abc_bank_level_performance.csv
abc_predictions_2021_2022.csv
abc_validation_class_distributions.csv
abc_xgboost_performance.csv
tb_900_parameter_screening.csv


**Class Weighting Experiment**

In [16]:

class_weight_results = []

for approach in candidate_configs:

    for country in ["Malaysia", "Singapore"]:

        subset = candidate_data.loc[
            (candidate_data["Approach"] == approach)
            & (candidate_data["Country"] == country)
        ]

        train = subset.loc[subset["Split"] == "Train"]
        valid = subset.loc[subset["Split"] == "Validation"]

        X_train = train[feature_columns]
        y_train = train["Label"].to_numpy(dtype=int) + 1

        X_valid = valid[feature_columns]
        y_valid = valid["Label"].to_numpy(dtype=int) + 1

        # Calculate inverse-frequency weights using training labels only.
        counts = np.bincount(y_train, minlength=3)

        assert (counts > 0).all()
        assert X_train.notna().all().all()
        assert X_valid.notna().all().all()

        balanced_weights = len(y_train) / (3.0 * counts)

        for alpha in [0.0, 0.5, 1.0]:

            class_weights = balanced_weights ** alpha
            sample_weights = class_weights[y_train]

            # Keep the average training weight equal to one.
            sample_weights = (
                sample_weights / sample_weights.mean()
            )

            assert np.isclose(sample_weights.mean(), 1.0)

            model = XGBClassifier(**model_params)

            model.fit(
                X_train,
                y_train,
                sample_weight=sample_weights,
                verbose=False
            )

            y_pred = model.predict(X_valid).astype(int)

            recalls = recall_score(
                y_valid, y_pred,
                labels=[0, 1, 2],
                average=None,
                zero_division=0
            )

            # Measure cases where Up is predicted as Down or vice versa.
            directional_actual = np.isin(y_valid, [0, 2])

            opposite_errors = (
                ((y_valid == 0) & (y_pred == 2))
                | ((y_valid == 2) & (y_pred == 0))
            )

            opposite_rate = (
                opposite_errors.sum() / directional_actual.sum()
            )

            # Precision among all predictions of Up or Down.
            directional_predicted = np.isin(y_pred, [0, 2])

            correct_directional = (
                (y_pred == y_valid) & directional_predicted
            )

            signal_precision = (
                correct_directional.sum()
                / directional_predicted.sum()
                if directional_predicted.any()
                else np.nan
            )

            class_weight_results.append({
                "Approach": approach,
                "Country": country,
                "Alpha": alpha,
                "Accuracy": accuracy_score(
                    y_valid, y_pred
                ) * 100,
                "Balanced_Accuracy": balanced_accuracy_score(
                    y_valid, y_pred
                ) * 100,
                "Macro_F1": f1_score(
                    y_valid, y_pred,
                    average="macro"
                ),
                "Down_Recall": recalls[0] * 100,
                "Flat_Recall": recalls[1] * 100,
                "Up_Recall": recalls[2] * 100,
                "Directional_Recall": (
                    recalls[0] + recalls[2]
                ) * 50,
                "Directional_Signal_Precision":
                    signal_precision * 100,
                "Opposite_Direction_Rate":
                    opposite_rate * 100,
                "Predicted_Flat_Pct":
                    (y_pred == 1).mean() * 100
            })

class_weight_results_df = pd.DataFrame(
    class_weight_results
)

# Verify the unweighted models reproduce Notebook 06.
for _, row in class_weight_results_df.loc[
    class_weight_results_df["Alpha"] == 0.0
].iterrows():

    original = results_df.loc[
        (results_df["Approach"] == row["Approach"])
        & (results_df["Country"] == row["Country"])
    ].iloc[0]

    assert np.isclose(
        row["Accuracy"],
        original["Accuracy"] * 100
    )

    assert np.isclose(
        row["Macro_F1"],
        original["Macro_F1"]
    )

    assert np.isclose(
        row["Directional_Recall"],
        original["Directional_Recall"] * 100
    )

print("=== CLASS WEIGHTING COMPARISON ===")

display(
    class_weight_results_df[
        [
            "Country", "Approach", "Alpha",
            "Accuracy", "Balanced_Accuracy",
            "Macro_F1", "Directional_Recall",
            "Flat_Recall",
            "Directional_Signal_Precision",
            "Opposite_Direction_Rate",
            "Predicted_Flat_Pct"
        ]
    ].sort_values(
        ["Country", "Approach", "Alpha"]
    ).round(4)
)

# Compare each weight setting against the same label configuration.
baseline = class_weight_results_df.loc[
    class_weight_results_df["Alpha"] == 0.0
].set_index(["Country", "Approach"])

comparison_rows = []

for _, row in class_weight_results_df.loc[
    class_weight_results_df["Alpha"] > 0.0
].iterrows():

    original = baseline.loc[
        (row["Country"], row["Approach"])
    ]

    comparison_rows.append({
        "Country": row["Country"],
        "Approach": row["Approach"],
        "Alpha": row["Alpha"],
        "Accuracy_Change_pp":
            row["Accuracy"] - original["Accuracy"],
        "Macro_F1_Change":
            row["Macro_F1"] - original["Macro_F1"],
        "Directional_Recall_Change_pp":
            row["Directional_Recall"]
            - original["Directional_Recall"],
        "Flat_Recall_Change_pp":
            row["Flat_Recall"] - original["Flat_Recall"],
        "Opposite_Error_Change_pp":
            row["Opposite_Direction_Rate"]
            - original["Opposite_Direction_Rate"]
    })

class_weight_changes_df = pd.DataFrame(comparison_rows)

print("\n=== CHANGE COMPARED WITH ORIGINAL UNWEIGHTED MODEL ===")

display(
    class_weight_changes_df.round(4)
)



=== CLASS WEIGHTING COMPARISON ===


,Country,Approach,Alpha,Accuracy,Balanced_Accuracy,Macro_F1,Directional_Recall,Flat_Recall,Directional_Signal_Precision,Opposite_Direction_Rate,Predicted_Flat_Pct
0,Malaysia,A_Original,0.0,54.0396,45.1297,0.4361,24.7914,85.8065,40.4575,22.6001,68.8627
1,Malaysia,A_Original,0.5,52.7710,46.0348,0.4562,30.7573,76.5899,36.6787,30.8222,56.9775
2,Malaysia,A_Original,1.0,49.9889,45.7742,0.4576,36.3110,64.7005,34.0615,37.1933,44.9811
6,Malaysia,B_Common,0.0,50.3672,49.6771,0.4881,40.7940,67.4433,41.8045,38.0154,38.0815
7,Malaysia,B_Common,0.5,49.9444,49.3110,0.4915,41.6244,64.6842,41.3134,38.8889,35.9448
8,Malaysia,B_Common,1.0,50.0556,49.5421,0.4975,43.7490,61.1281,41.8030,38.7491,33.3408
12,Malaysia,C_Market_Specific,0.0,52.2591,52.4726,0.5145,43.1861,71.0458,42.2871,43.1320,33.6301
13,Malaysia,C_Market_Specific,0.5,53.0381,53.0889,0.5271,43.8817,71.5033,43.2115,42.4570,33.6078
14,Malaysia,C_Market_Specific,1.0,52.8155,52.6973,0.5260,43.0983,71.8954,43.0068,42.4232,34.1197
3,Singapore,A_Original,0.0,48.4111,43.3403,0.4303,29.7731,70.4748,33.8959,33.6646,51.9270



=== CHANGE COMPARED WITH ORIGINAL UNWEIGHTED MODEL ===


,Country,Approach,Alpha,Accuracy_Change_pp,Macro_F1_Change,Directional_Recall_Change_pp,Flat_Recall_Change_pp,Opposite_Error_Change_pp
0,Malaysia,A_Original,0.5,-1.2686,0.0201,5.9659,-9.2166,8.2221
1,Malaysia,A_Original,1.0,-4.0507,0.0216,11.5197,-21.1060,14.5932
2,Singapore,A_Original,0.5,-0.8790,0.0022,2.1831,-4.4510,2.1118
3,Singapore,A_Original,1.0,-1.5551,0.0074,5.3503,-9.6439,3.9752
4,Malaysia,B_Common,0.5,-0.4229,0.0034,0.8304,-2.7590,0.8735
5,Malaysia,B_Common,1.0,-0.3116,0.0094,2.9550,-6.3151,0.7338
6,Singapore,B_Common,0.5,0.4057,0.0046,-0.3568,2.4845,-0.6024
7,Singapore,B_Common,1.0,1.0142,0.0111,0.3312,3.7267,-2.4096
8,Malaysia,C_Market_Specific,0.5,0.7790,0.0126,0.6956,0.4575,-0.6750
9,Malaysia,C_Market_Specific,1.0,0.5564,0.0115,-0.0878,0.8497,-0.7087


**Check whether the improvements hold in both years**

In [18]:

weight_audit_candidates = [
    ("Malaysia", "C_Market_Specific", 0.5),
    ("Singapore", "B_Common", 1.0),
    ("Singapore", "C_Market_Specific", 1.0)
]

weight_year_results = []

for country, approach, selected_alpha in weight_audit_candidates:

    subset = candidate_data.loc[
        (candidate_data["Country"] == country)
        & (candidate_data["Approach"] == approach)
    ]

    train = subset.loc[subset["Split"] == "Train"]
    valid = subset.loc[subset["Split"] == "Validation"]

    X_train = train[feature_columns]
    y_train = train["Label"].to_numpy(dtype=int) + 1

    X_valid = valid[feature_columns]
    y_valid = valid["Label"].to_numpy(dtype=int) + 1

    counts = np.bincount(y_train, minlength=3)

    assert (counts > 0).all()

    balanced_weights = len(y_train) / (3.0 * counts)

    for alpha in [0.0, selected_alpha]:

        # Calculate weights using training data only.
        sample_weights = balanced_weights[y_train] ** alpha
        sample_weights /= sample_weights.mean()

        model = XGBClassifier(**model_params)

        model.fit(
            X_train,
            y_train,
            sample_weight=sample_weights,
            verbose=False
        )

        predictions = model.predict(X_valid).astype(int)

        for year in [2021, 2022]:

            year_mask = valid["Date"].dt.year.to_numpy() == year

            yt = y_valid[year_mask]
            yp = predictions[year_mask]

            assert len(yt) > 0

            recalls = recall_score(
                yt, yp,
                labels=[0, 1, 2],
                average=None,
                zero_division=0
            )

            actual_directional = np.isin(yt, [0, 2])
            predicted_directional = np.isin(yp, [0, 2])

            opposite_errors = (
                ((yt == 0) & (yp == 2))
                | ((yt == 2) & (yp == 0))
            )

            correct_signals = (
                predicted_directional & (yt == yp)
            )

            opposite_rate = (
                100 * opposite_errors.sum()
                / actual_directional.sum()
                if actual_directional.any()
                else np.nan
            )

            signal_precision = (
                100 * correct_signals.sum()
                / predicted_directional.sum()
                if predicted_directional.any()
                else np.nan
            )

            weight_year_results.append({
                "Country": country,
                "Approach": approach,
                "Year": year,
                "Alpha": alpha,
                "Rows": len(yt),
                "Accuracy": accuracy_score(yt, yp) * 100,
                "Balanced_Accuracy":
                    balanced_accuracy_score(yt, yp) * 100,
                "Macro_F1": f1_score(
                    yt, yp, average="macro", zero_division=0
                ),
                "Directional_Recall":
                    (recalls[0] + recalls[2]) * 50,
                "Flat_Recall": recalls[1] * 100,
                "Signal_Precision": signal_precision,
                "Opposite_Direction_Rate": opposite_rate
            })

weight_year_df = pd.DataFrame(weight_year_results)

print("=== YEARLY CLASS WEIGHTING PERFORMANCE ===")
display(weight_year_df.round(4))

# Compare weighted and unweighted models within the same year
baseline = weight_year_df.loc[
    weight_year_df["Alpha"] == 0.0
].set_index(["Country", "Approach", "Year"])

weighted = weight_year_df.loc[
    weight_year_df["Alpha"] > 0.0
].copy()

change_rows = []

for _, row in weighted.iterrows():

    original = baseline.loc[
        (row["Country"], row["Approach"], row["Year"])
    ]

    change_rows.append({
        "Country": row["Country"],
        "Approach": row["Approach"],
        "Year": row["Year"],
        "Alpha": row["Alpha"],
        "Accuracy_Change_pp":
            row["Accuracy"] - original["Accuracy"],
        "Macro_F1_Change":
            row["Macro_F1"] - original["Macro_F1"],
        "Directional_Recall_Change_pp":
            row["Directional_Recall"]
            - original["Directional_Recall"],
        "Opposite_Error_Change_pp":
            row["Opposite_Direction_Rate"]
            - original["Opposite_Direction_Rate"]
    })

weight_year_changes_df = pd.DataFrame(change_rows)

print("\n=== YEARLY CHANGE FROM UNWEIGHTED BASELINE ===")
display(weight_year_changes_df.round(4))




=== YEARLY CLASS WEIGHTING PERFORMANCE ===


,Country,Approach,Year,Alpha,Rows,Accuracy,Balanced_Accuracy,Macro_F1,Directional_Recall,Flat_Recall,Signal_Precision,Opposite_Direction_Rate
0,Malaysia,C_Market_Specific,2021,0.0,2235,51.6331,51.4440,0.5012,42.0610,70.2100,41.6555,43.4487
1,Malaysia,C_Market_Specific,2022,0.0,2258,52.8787,53.2975,0.5249,44.0088,71.8750,42.9144,42.8188
2,Malaysia,C_Market_Specific,2021,0.5,2235,51.9016,51.7021,0.5108,42.1857,70.7349,41.9028,43.3130
3,Malaysia,C_Market_Specific,2022,0.5,2258,54.1630,54.2448,0.5404,45.2344,72.2656,44.5037,41.6107
4,Singapore,B_Common,2021,0.0,759,49.5389,49.1623,0.4942,43.7434,60.0000,41.4914,40.4858
5,Singapore,B_Common,2022,0.0,720,43.7500,43.8398,0.4383,38.2368,55.0459,37.0019,46.6135
6,Singapore,B_Common,2021,1.0,759,49.2754,49.3524,0.4909,42.8966,62.2642,41.3861,39.6761
7,Singapore,B_Common,2022,1.0,720,46.1111,46.6011,0.4653,39.6265,60.5505,40.0000,42.6295
8,Singapore,C_Market_Specific,2021,0.0,759,46.2451,44.3812,0.4424,35.1959,62.7517,37.0203,36.4425
9,Singapore,C_Market_Specific,2022,0.0,720,42.5000,42.5417,0.4221,36.4621,54.7009,39.1209,35.1852



=== YEARLY CHANGE FROM UNWEIGHTED BASELINE ===


,Country,Approach,Year,Alpha,Accuracy_Change_pp,Macro_F1_Change,Directional_Recall_Change_pp,Opposite_Error_Change_pp
0,Malaysia,C_Market_Specific,2021,0.5,0.2685,0.0096,0.1247,-0.1358
1,Malaysia,C_Market_Specific,2022,0.5,1.2843,0.0154,1.2256,-1.2081
2,Singapore,B_Common,2021,1.0,-0.2635,-0.0033,-0.8469,-0.8097
3,Singapore,B_Common,2022,1.0,2.3611,0.0271,1.3897,-3.9841
4,Singapore,C_Market_Specific,2021,1.0,1.3175,0.0141,1.4578,-1.9523
5,Singapore,C_Market_Specific,2022,1.0,2.0833,0.0218,2.1825,-2.2634


**Class Weighting Performance Tables**

In [20]:

weighted_detailed_results = []

for country in ["Malaysia", "Singapore"]:
    for approach in candidate_configs:

        subset = candidate_data.loc[
            (candidate_data["Country"] == country)
            & (candidate_data["Approach"] == approach)
        ]

        train = subset.loc[subset["Split"] == "Train"]
        valid = subset.loc[subset["Split"] == "Validation"]

        X_train = train[feature_columns]
        y_train = train["Label"].to_numpy(dtype=int) + 1

        X_valid = valid[feature_columns]
        y_valid = valid["Label"].to_numpy(dtype=int) + 1

        counts = np.bincount(y_train, minlength=3)
        assert (counts > 0).all()

        balanced_weights = len(y_train) / (3.0 * counts)

        # Use the majority class from training data as the baseline.
        majority_class = np.argmax(counts)
        baseline_predictions = np.full(len(y_valid), majority_class)

        for alpha in [0.0, 0.5, 1.0]:

            sample_weights = balanced_weights[y_train] ** alpha
            sample_weights /= sample_weights.mean()

            model = XGBClassifier(**model_params)

            model.fit(
                X_train,
                y_train,
                sample_weight=sample_weights,
                verbose=False
            )

            y_pred = model.predict(X_valid).astype(int)

            recalls = recall_score(
                y_valid,
                y_pred,
                labels=[0, 1, 2],
                average=None,
                zero_division=0
            )

            precisions = precision_score(
                y_valid,
                y_pred,
                labels=[0, 1, 2],
                average=None,
                zero_division=0
            )

            directional_actual = np.isin(y_valid, [0, 2])
            directional_predicted = np.isin(y_pred, [0, 2])

            opposite_errors = (
                ((y_valid == 0) & (y_pred == 2))
                | ((y_valid == 2) & (y_pred == 0))
            )

            correct_signals = (
                directional_predicted & (y_valid == y_pred)
            )

            weighted_detailed_results.append({
                "Country": country,
                "Approach": approach,
                "Alpha": alpha,
                "Accuracy": accuracy_score(y_valid, y_pred) * 100,
                "Baseline_Accuracy": accuracy_score(
                    y_valid, baseline_predictions
                ) * 100,
                "Balanced_Accuracy":
                    balanced_accuracy_score(y_valid, y_pred) * 100,
                "Macro_F1": f1_score(
                    y_valid, y_pred, average="macro"
                ),
                "MCC": matthews_corrcoef(y_valid, y_pred),
                "Down_Recall": recalls[0] * 100,
                "Flat_Recall": recalls[1] * 100,
                "Up_Recall": recalls[2] * 100,
                "Directional_Recall":
                    (recalls[0] + recalls[2]) * 50,
                "Down_Precision": precisions[0] * 100,
                "Up_Precision": precisions[2] * 100,
                "Signal_Precision":
                    correct_signals.sum()
                    / directional_predicted.sum() * 100
                    if directional_predicted.any() else np.nan,
                "Opposite_Error":
                    opposite_errors.sum()
                    / directional_actual.sum() * 100
                    if directional_actual.any() else np.nan
            })

weighted_detailed_df = pd.DataFrame(
    weighted_detailed_results
)

# Keep the same metric order as the original Triple Barrier tables.
metric_labels = {
    "Accuracy (%)": "Accuracy",
    "Majority baseline accuracy (%)": "Baseline_Accuracy",
    "Balanced accuracy (%)": "Balanced_Accuracy",
    "Macro F1": "Macro_F1",
    "MCC": "MCC",
    "Down recall (%)": "Down_Recall",
    "Flat recall (%)": "Flat_Recall",
    "Up recall (%)": "Up_Recall",
    "Directional recall (%)": "Directional_Recall",
    "Down precision (%)": "Down_Precision",
    "Up precision (%)": "Up_Precision",
    "Directional signal precision (%)": "Signal_Precision",
    "Opposite-direction error (%)": "Opposite_Error"
}

approach_names = {
    "A_Original": "A — Original",
    "B_Common": "B — Common",
    "C_Market_Specific": "C — Market-specific"
}

# Display one Word-friendly table per market and weighting level.
for country in ["Malaysia", "Singapore"]:

    for alpha in [0.0, 0.5, 1.0]:

        table = pd.DataFrame({
            "Metric": list(metric_labels.keys())
        })

        for approach, display_name in approach_names.items():

            result = weighted_detailed_df.loc[
                (weighted_detailed_df["Country"] == country)
                & (weighted_detailed_df["Approach"] == approach)
                & (weighted_detailed_df["Alpha"] == alpha)
            ].iloc[0]

            table[display_name] = [
                (
                    f"{result[column]:.4f}"
                    if column in ["Macro_F1", "MCC"]
                    else f"{result[column]:.2f}"
                )
                for column in metric_labels.values()
            ]

        print(
            f"\n=== {country.upper()} — CLASS WEIGHTING "
            f"α = {alpha:.1f} ==="
        )

        display(table)



=== MALAYSIA — CLASS WEIGHTING α = 0.0 ===


,Metric,A — Original,B — Common,C — Market-specific
0,Accuracy (%),54.04,50.37,52.26
1,Majority baseline accuracy (%),48.30,36.30,31.58
2,Balanced accuracy (%),45.13,49.68,52.47
3,Macro F1,0.4361,0.4881,0.5145
4,MCC,0.2349,0.2573,0.2928
5,Down recall (%),32.24,53.36,58.07
6,Flat recall (%),85.81,67.44,71.05
7,Up recall (%),17.35,28.23,28.30
8,Directional recall (%),24.79,40.79,43.19
9,Down precision (%),39.49,42.24,42.28



=== MALAYSIA — CLASS WEIGHTING α = 0.5 ===


,Metric,A — Original,B — Common,C — Market-specific
0,Accuracy (%),52.77,49.94,53.04
1,Majority baseline accuracy (%),48.30,36.30,31.58
2,Balanced accuracy (%),46.03,49.31,53.09
3,Macro F1,0.4562,0.4915,0.5271
4,MCC,0.2339,0.2491,0.2995
5,Down recall (%),34.89,47.98,52.85
6,Flat recall (%),76.59,64.68,71.50
7,Up recall (%),26.63,35.27,34.91
8,Directional recall (%),30.76,41.62,43.88
9,Down precision (%),36.00,41.60,42.88



=== MALAYSIA — CLASS WEIGHTING α = 1.0 ===


,Metric,A — Original,B — Common,C — Market-specific
0,Accuracy (%),49.99,50.06,52.82
1,Majority baseline accuracy (%),48.30,36.30,31.58
2,Balanced accuracy (%),45.77,49.54,52.70
3,Macro F1,0.4576,0.4975,0.5260
4,MCC,0.2192,0.2510,0.2928
5,Down recall (%),37.44,44.02,46.30
6,Flat recall (%),64.70,61.13,71.90
7,Up recall (%),35.18,43.48,39.90
8,Directional recall (%),36.31,43.75,43.10
9,Down precision (%),34.17,42.43,42.75


... (2 outputs left out)


=== SINGAPORE — CLASS WEIGHTING α = 0.5 ===


,Metric,A — Original,B — Common,C — Market-specific
0,Accuracy (%),47.53,47.13,45.37
1,Majority baseline accuracy (%),45.57,36.65,34.48
2,Balanced accuracy (%),43.31,47.00,44.64
3,Macro F1,0.4325,0.4724,0.4443
4,MCC,0.1799,0.2044,0.1753
5,Down recall (%),30.03,35.90,34.55
6,Flat recall (%),66.02,60.25,60.15
7,Up recall (%),33.89,44.83,39.22
8,Directional recall (%),31.96,40.37,36.88
9,Down precision (%),33.63,34.98,39.12



=== SINGAPORE — CLASS WEIGHTING α = 1.0 ===


,Metric,A — Original,B — Common,C — Market-specific
0,Accuracy (%),46.86,47.73,46.11
1,Majority baseline accuracy (%),45.57,36.65,34.48
2,Balanced accuracy (%),43.69,47.87,45.51
3,Macro F1,0.4377,0.4789,0.4520
4,MCC,0.1816,0.2157,0.1876
5,Down recall (%),34.46,40.97,37.99
6,Flat recall (%),60.83,61.49,61.28
7,Up recall (%),35.78,41.14,37.25
8,Directional recall (%),35.12,41.06,37.62
9,Down precision (%),33.76,37.65,40.19


**Random Forest Feature Importance**

In [21]:

from sklearn.ensemble import RandomForestClassifier

rf_importance_results = []

for country in ["Malaysia", "Singapore"]:

    for approach in candidate_configs:

        train = candidate_data.loc[
            (candidate_data["Country"] == country)
            & (candidate_data["Approach"] == approach)
            & (candidate_data["Split"] == "Train")
        ]

        X_train = train[feature_columns]
        y_train = train["Label"].to_numpy(dtype=int)

        assert X_train.notna().all().all()
        assert set(np.unique(y_train)) == {-1, 0, 1}

        # Keep Random Forest settings consistent across configurations.
        rf_model = RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=5,
            max_features="sqrt",
            class_weight="balanced_subsample",
            random_state=42,
            n_jobs=-1
        )

        rf_model.fit(X_train, y_train)

        importances = rf_model.feature_importances_

        assert np.isclose(importances.sum(), 1.0)

        for feature, importance in zip(
            feature_columns, importances
        ):

            rf_importance_results.append({
                "Country": country,
                "Approach": approach,
                "Feature": feature,
                "Importance": importance
            })

rf_importance_df = pd.DataFrame(rf_importance_results)

# Rank the 23 indicators separately for each model.
rf_importance_df["Rank"] = (
    rf_importance_df
    .groupby(["Country", "Approach"])["Importance"]
    .rank(method="first", ascending=False)
    .astype(int)
)

rf_importance_df = rf_importance_df.sort_values(
    ["Country", "Approach", "Rank"]
).reset_index(drop=True)

print("=== TOP 8 RANDOM FOREST FEATURES BY CONFIGURATION ===")

display(
    rf_importance_df.loc[
        rf_importance_df["Rank"] <= 8,
        ["Country", "Approach", "Rank", "Feature", "Importance"]
    ].round(4)
)

# Summarise indicator rankings across A, B and C for each market.
rf_rank_summary = (
    rf_importance_df
    .groupby(["Country", "Feature"])
    .agg(
        Mean_Rank=("Rank", "mean"),
        Mean_Importance=("Importance", "mean"),
        Top_5_Count=("Rank", lambda x: int((x <= 5).sum()))
    )
    .reset_index()
    .sort_values(
        ["Country", "Mean_Rank", "Mean_Importance"],
        ascending=[True, True, False]
    )
)

print("\n=== TOP 10 FEATURES ACROSS CONFIGURATIONS ===")

display(
    rf_rank_summary.groupby("Country")
    .head(10)
    .round(4)
)

rf_importance_df.to_csv(
    output_dir / "rf_feature_importance_abc.csv",
    index=False
)

rf_rank_summary.to_csv(
    output_dir / "rf_feature_rank_summary.csv",
    index=False
)


=== TOP 8 RANDOM FOREST FEATURES BY CONFIGURATION ===


,Country,Approach,Rank,Feature,Importance
0,Malaysia,A_Original,1,volatility_5,0.1270
1,Malaysia,A_Original,2,volatility_10,0.0551
2,Malaysia,A_Original,3,volatility_20,0.0539
3,Malaysia,A_Original,4,skewness_20,0.0473
4,Malaysia,A_Original,5,macd_signal,0.0473
5,Malaysia,A_Original,6,bb_width,0.0465
6,Malaysia,A_Original,7,macd,0.0418
7,Malaysia,A_Original,8,macd_diff,0.0411
23,Malaysia,B_Common,1,returns_lag_1,0.1120
24,Malaysia,B_Common,2,returns,0.0794



=== TOP 10 FEATURES ACROSS CONFIGURATIONS ===


,Country,Feature,Mean_Rank,Mean_Importance,Top_5_Count
20,Malaysia,volatility_5,2.3333,0.0830,3
11,Malaysia,returns_lag_1,5.6667,0.0916,2
19,Malaysia,volatility_20,6.0000,0.0445,1
18,Malaysia,volatility_10,6.3333,0.0441,1
14,Malaysia,skewness_20,7.3333,0.0417,1
10,Malaysia,returns,8.6667,0.0690,2
4,Malaysia,daily_range,8.6667,0.0395,0
17,Malaysia,sma_5_ratio,9.3333,0.0410,2
3,Malaysia,candle_body,9.6667,0.0451,2
2,Malaysia,bb_width,9.6667,0.0406,0


**Feature-Group Ablation**

In [22]:

feature_groups = {
    "Returns": [
        "returns", "returns_lag_1", "returns_lag_2"
    ],
    "Momentum": [
        "momentum_5", "momentum_10"
    ],
    "Candlestick": [
        "daily_range", "candle_body"
    ],
    "Moving_Averages": [
        "sma_5_ratio", "sma_10_ratio", "sma_20_ratio"
    ],
    "Volatility": [
        "volatility_5", "volatility_10", "volatility_20"
    ],
    "RSI": ["rsi"],
    "MACD": [
        "macd", "macd_signal", "macd_diff"
    ],
    "Volume": [
        "volume_change", "volume_ratio_20"
    ],
    "Bollinger_Bands": [
        "bb_high", "bb_low", "bb_width"
    ],
    "Skewness": ["skewness_20"]
}

assert set(
    feature for group in feature_groups.values()
    for feature in group
) == set(feature_columns)

ablation_candidates = [
    ("Malaysia", "C_Market_Specific", 0.5),
    ("Singapore", "B_Common", 1.0),
    ("Singapore", "C_Market_Specific", 1.0)
]

feature_ablation_results = []

for country, approach, alpha in ablation_candidates:

    subset = candidate_data.loc[
        (candidate_data["Country"] == country)
        & (candidate_data["Approach"] == approach)
    ]

    train = subset.loc[subset["Split"] == "Train"]
    valid = subset.loc[subset["Split"] == "Validation"]

    y_train = train["Label"].to_numpy(dtype=int) + 1
    y_valid = valid["Label"].to_numpy(dtype=int) + 1

    counts = np.bincount(y_train, minlength=3)
    assert (counts > 0).all()

    # Calculate class weights from training observations only.
    balanced_weights = len(y_train) / (3.0 * counts)
    sample_weights = balanced_weights[y_train] ** alpha
    sample_weights /= sample_weights.mean()

    # First fit the full-feature baseline, then remove one group at a time.
    experiments = {"All_Features": feature_columns.copy()}

    for group_name, group_features in feature_groups.items():
        experiments[f"Remove_{group_name}"] = [
            feature for feature in feature_columns
            if feature not in group_features
        ]

    for experiment, selected_features in experiments.items():

        model = XGBClassifier(**model_params)

        model.fit(
            train[selected_features],
            y_train,
            sample_weight=sample_weights,
            verbose=False
        )

        y_pred = model.predict(
            valid[selected_features]
        ).astype(int)

        recalls = recall_score(
            y_valid, y_pred,
            labels=[0, 1, 2],
            average=None,
            zero_division=0
        )

        directional_actual = np.isin(y_valid, [0, 2])

        opposite_errors = (
            ((y_valid == 0) & (y_pred == 2))
            | ((y_valid == 2) & (y_pred == 0))
        )

        opposite_rate = (
            100 * opposite_errors.sum()
            / directional_actual.sum()
        )

        feature_ablation_results.append({
            "Country": country,
            "Approach": approach,
            "Alpha": alpha,
            "Experiment": experiment,
            "Features": len(selected_features),
            "Accuracy": accuracy_score(
                y_valid, y_pred
            ) * 100,
            "Balanced_Accuracy":
                balanced_accuracy_score(y_valid, y_pred) * 100,
            "Macro_F1": f1_score(
                y_valid, y_pred,
                average="macro",
                zero_division=0
            ),
            "Down_Recall": recalls[0] * 100,
            "Flat_Recall": recalls[1] * 100,
            "Up_Recall": recalls[2] * 100,
            "Directional_Recall":
                (recalls[0] + recalls[2]) * 50,
            "Opposite_Error": opposite_rate
        })

feature_ablation_df = pd.DataFrame(
    feature_ablation_results
)

# Compare every ablation with its own full-feature baseline.
baseline_df = feature_ablation_df.loc[
    feature_ablation_df["Experiment"] == "All_Features"
].set_index(["Country", "Approach"])

for metric in [
    "Accuracy",
    "Balanced_Accuracy",
    "Macro_F1",
    "Down_Recall",
    "Flat_Recall",
    "Up_Recall",
    "Directional_Recall",
    "Opposite_Error"
]:

    baseline_values = [
        baseline_df.loc[
            (row["Country"], row["Approach"]),
            metric
        ]
        for _, row in feature_ablation_df.iterrows()
    ]

    feature_ablation_df[f"{metric}_Change"] = (
        feature_ablation_df[metric].to_numpy()
        - np.asarray(baseline_values)
    )

# Verify the full-feature models reproduce the class-weighting results.
for country, approach, alpha in ablation_candidates:

    actual = feature_ablation_df.loc[
        (feature_ablation_df["Country"] == country)
        & (feature_ablation_df["Approach"] == approach)
        & (feature_ablation_df["Experiment"] == "All_Features")
    ].iloc[0]

    expected = class_weight_results_df.loc[
        (class_weight_results_df["Country"] == country)
        & (class_weight_results_df["Approach"] == approach)
        & (class_weight_results_df["Alpha"] == alpha)
    ].iloc[0]

    assert np.isclose(
        actual["Macro_F1"], expected["Macro_F1"]
    )

    assert np.isclose(
        actual["Accuracy"], expected["Accuracy"]
    )

for country, approach, alpha in ablation_candidates:

    results = feature_ablation_df.loc[
        (feature_ablation_df["Country"] == country)
        & (feature_ablation_df["Approach"] == approach)
    ]

    print(
        f"\n=== {country.upper()} — "
        f"{approach} (α = {alpha}) ==="
    )

    display(
        results[
            [
                "Experiment",
                "Features",
                "Accuracy",
                "Macro_F1",
                "Down_Recall",
                "Flat_Recall",
                "Up_Recall",
                "Directional_Recall",
                "Opposite_Error",
                "Macro_F1_Change",
                "Directional_Recall_Change",
                "Opposite_Error_Change"
            ]
        ].round(4)
    )

feature_ablation_df.to_csv(
    output_dir / "feature_group_ablation.csv",
    index=False
)



=== MALAYSIA — C_Market_Specific (α = 0.5) ===


,Experiment,Features,Accuracy,Macro_F1,Down_Recall,Flat_Recall,Up_Recall,Directional_Recall,Opposite_Error,Macro_F1_Change,Directional_Recall_Change,Opposite_Error_Change
0,All_Features,23,53.0381,0.5271,52.8541,71.5033,34.9093,43.8817,42.4570,0.0000,0.0000,0.0000
1,Remove_Returns,20,44.2911,0.4329,42.8471,62.6144,27.4611,35.1541,35.9096,-0.0942,-8.7276,-6.5474
2,Remove_Momentum,21,52.7932,0.5232,54.4045,71.4379,32.8368,43.6206,42.4232,-0.0039,-0.2611,-0.0337
3,Remove_Candlestick,21,52.9268,0.5235,55.3206,72.1569,31.6710,43.4958,42.6932,-0.0036,-0.3859,0.2362
4,Remove_Moving_Averages,20,53.1493,0.5266,55.4616,71.5686,32.7720,44.1168,42.2207,-0.0005,0.2351,-0.2362
5,Remove_Volatility,20,51.8584,0.5157,54.7569,69.0850,32.1244,43.4406,43.7057,-0.0114,-0.4411,1.2487
6,Remove_RSI,22,53.0826,0.5260,53.8407,72.1569,33.4845,43.6626,42.2882,-0.0011,-0.2191,-0.1687
7,Remove_MACD,20,53.2829,0.5248,58.2100,72.2222,29.9870,44.0985,41.8495,-0.0023,0.2168,-0.6075
8,Remove_Volume,21,52.9045,0.5243,53.9112,72.1569,32.9016,43.4064,42.7945,-0.0028,-0.4753,0.3375
9,Remove_Bollinger_Bands,20,53.0158,0.5245,54.7569,71.8301,32.7720,43.7644,41.5120,-0.0026,-0.1173,-0.9450



=== SINGAPORE — B_Common (α = 1.0) ===


,Experiment,Features,Accuracy,Macro_F1,Down_Recall,Flat_Recall,Up_Recall,Directional_Recall,Opposite_Error,Macro_F1_Change,Directional_Recall_Change,Opposite_Error_Change
11,All_Features,23,47.7350,0.4789,40.9692,61.4907,41.1439,41.0565,41.1647,0.0000,0.0000,0.0000
12,Remove_Returns,20,38.0663,0.3781,32.3789,49.0683,33.0258,32.7023,35.0402,-0.1007,-8.3542,-6.1245
13,Remove_Momentum,21,47.1264,0.4723,38.9868,62.3188,40.4059,39.6963,42.2691,-0.0066,-1.3602,1.1044
14,Remove_Candlestick,21,47.2617,0.4733,37.8855,62.5259,41.5129,39.6992,42.3695,-0.0056,-1.3573,1.2048
15,Remove_Moving_Averages,20,47.3969,0.4757,38.9868,62.1118,41.3284,40.1576,42.7711,-0.0032,-0.8989,1.6064
16,Remove_Volatility,20,48.8844,0.4912,42.0705,61.9048,42.9889,42.5297,41.1647,0.0123,1.4732,0.0000
17,Remove_RSI,22,47.2617,0.4738,39.4273,60.8696,41.6974,40.5624,41.2651,-0.0051,-0.4942,0.1004
18,Remove_MACD,20,47.0588,0.4715,38.9868,60.4555,41.8819,40.4344,40.9639,-0.0074,-0.6222,-0.2008
19,Remove_Volume,21,47.8702,0.4793,38.9868,62.9400,41.8819,40.4344,41.5663,0.0004,-0.6222,0.4016
20,Remove_Bollinger_Bands,20,46.8560,0.4693,37.2247,60.8696,42.4354,39.8300,42.0683,-0.0096,-1.2265,0.9036



=== SINGAPORE — C_Market_Specific (α = 1.0) ===


,Experiment,Features,Accuracy,Macro_F1,Down_Recall,Flat_Recall,Up_Recall,Directional_Recall,Opposite_Error,Macro_F1_Change,Directional_Recall_Change,Opposite_Error_Change
22,All_Features,23,46.1122,0.4520,37.9863,61.2782,37.2549,37.6206,33.6853,0.0000,0.0000,0.0000
23,Remove_Returns,20,44.4219,0.4348,35.0114,59.3985,36.8627,35.9371,34.1077,-0.0173,-1.6835,0.4224
24,Remove_Momentum,21,44.2191,0.4317,34.3249,60.7143,35.4902,34.9076,35.9029,-0.0204,-2.7130,2.2175
25,Remove_Candlestick,21,44.7600,0.4373,34.3249,60.5263,37.2549,35.7899,35.1637,-0.0147,-1.8307,1.4784
26,Remove_Moving_Averages,20,44.3543,0.4336,34.0961,59.9624,36.8627,35.4794,35.2693,-0.0185,-2.1412,1.5839
27,Remove_Volatility,20,43.4077,0.4348,41.4188,45.8647,42.5490,41.9839,38.6484,-0.0173,4.3633,4.9630
28,Remove_RSI,22,44.8952,0.4399,36.3844,59.7744,36.6667,36.5256,34.2133,-0.0121,-1.0950,0.5280
29,Remove_MACD,20,44.9628,0.4387,33.4096,61.2782,37.8431,35.6264,35.2693,-0.0134,-1.9942,1.5839
30,Remove_Volume,21,44.3543,0.4312,32.2654,61.8421,36.4706,34.3680,35.7973,-0.0208,-3.2526,2.1119
31,Remove_Bollinger_Bands,20,45.3009,0.4435,34.7826,60.5263,38.4314,36.6070,36.0084,-0.0085,-1.0136,2.3231


In [23]:

country = "Singapore"
approach = "B_Common"
alpha = 1.0

subset = candidate_data.loc[
    (candidate_data["Country"] == country)
    & (candidate_data["Approach"] == approach)
]

train = subset.loc[subset["Split"] == "Train"]
valid = subset.loc[subset["Split"] == "Validation"]

# Identify the validation year without changing the data split.
column_lookup = {str(c).lower(): c for c in valid.columns}

if "year" in column_lookup:
    validation_years = valid[
        column_lookup["year"]
    ].astype(int).to_numpy()

elif "date" in column_lookup:
    validation_years = pd.to_datetime(
        valid[column_lookup["date"]]
    ).dt.year.to_numpy()

elif isinstance(valid.index, pd.DatetimeIndex):
    validation_years = valid.index.year.to_numpy()

else:
    raise KeyError("Validation date/year column not found.")

y_train = train["Label"].to_numpy(dtype=int) + 1
y_valid = valid["Label"].to_numpy(dtype=int) + 1

# Reproduce the class weighting used for Singapore B.
counts = np.bincount(y_train, minlength=3)
balanced_weights = len(y_train) / (3.0 * counts)

sample_weights = balanced_weights[y_train] ** alpha
sample_weights /= sample_weights.mean()

volatility_features = [
    "volatility_5",
    "volatility_10",
    "volatility_20"
]

feature_sets = {
    "All_23": list(feature_columns),
    "No_Volatility": [
        f for f in feature_columns
        if f not in volatility_features
    ]
}

yearly_ablation_results = []

for feature_set_name, selected_features in feature_sets.items():

    model = XGBClassifier(**model_params)

    model.fit(
        train[selected_features],
        y_train,
        sample_weight=sample_weights,
        verbose=False
    )

    predictions = model.predict(
        valid[selected_features]
    ).astype(int)

    # Evaluate separately by year and over the full validation period.
    periods = {
        "2021": validation_years == 2021,
        "2022": validation_years == 2022,
        "2021-2022": np.isin(validation_years, [2021, 2022])
    }

    for period, mask in periods.items():

        actual = y_valid[mask]
        predicted = predictions[mask]

        recalls = recall_score(
            actual,
            predicted,
            labels=[0, 1, 2],
            average=None,
            zero_division=0
        )

        directional_actual = np.isin(actual, [0, 2])

        opposite_errors = (
            ((actual == 0) & (predicted == 2))
            | ((actual == 2) & (predicted == 0))
        )

        yearly_ablation_results.append({
            "Period": period,
            "Feature_Set": feature_set_name,
            "Rows": len(actual),
            "Accuracy": accuracy_score(
                actual, predicted
            ) * 100,
            "Balanced_Accuracy": balanced_accuracy_score(
                actual, predicted
            ) * 100,
            "Macro_F1": f1_score(
                actual, predicted,
                average="macro",
                zero_division=0
            ),
            "Down_Recall": recalls[0] * 100,
            "Flat_Recall": recalls[1] * 100,
            "Up_Recall": recalls[2] * 100,
            "Directional_Recall":
                (recalls[0] + recalls[2]) * 50,
            "Opposite_Error":
                opposite_errors.sum()
                / directional_actual.sum() * 100
        })

yearly_ablation_df = pd.DataFrame(yearly_ablation_results)

# Verify that the full-period scores match the earlier ablation.
expected_f1 = {
    "All_23": 0.4789,
    "No_Volatility": 0.4912
}

for feature_set_name, expected in expected_f1.items():
    actual_f1 = yearly_ablation_df.loc[
        (yearly_ablation_df["Period"] == "2021-2022")
        & (yearly_ablation_df["Feature_Set"] == feature_set_name),
        "Macro_F1"
    ].iloc[0]

    assert np.isclose(actual_f1, expected, atol=0.00005)

print("=== SINGAPORE B — YEARLY VOLATILITY ABLATION ===")

display(yearly_ablation_df.round(4))

# Calculate the yearly difference after removing volatility.
comparison = yearly_ablation_df.pivot(
    index="Period",
    columns="Feature_Set",
    values=[
        "Accuracy",
        "Balanced_Accuracy",
        "Macro_F1",
        "Down_Recall",
        "Up_Recall",
        "Directional_Recall",
        "Opposite_Error"
    ]
)

changes = pd.DataFrame({
    metric: (
        comparison[(metric, "No_Volatility")]
        - comparison[(metric, "All_23")]
    )
    for metric in [
        "Accuracy",
        "Balanced_Accuracy",
        "Macro_F1",
        "Down_Recall",
        "Up_Recall",
        "Directional_Recall",
        "Opposite_Error"
    ]
})

print("\n=== CHANGE AFTER REMOVING VOLATILITY ===")
display(changes.round(4))

yearly_ablation_df.to_csv(
    output_dir / "sg_b_volatility_yearly_comparison.csv",
    index=False
)


=== SINGAPORE B — YEARLY VOLATILITY ABLATION ===


,Period,Feature_Set,Rows,Accuracy,Balanced_Accuracy,Macro_F1,Down_Recall,Flat_Recall,Up_Recall,Directional_Recall,Opposite_Error
0,2021,All_23,759,49.2754,49.3524,0.4909,47.4419,62.2642,38.3513,42.8966,39.6761
1,2022,All_23,720,46.1111,46.6011,0.4653,35.1464,60.5505,44.1065,39.6265,42.6295
2,2021-2022,All_23,1479,47.7350,47.8679,0.4789,40.9692,61.4907,41.1439,41.0565,41.1647
3,2021,No_Volatility,759,48.3531,48.3572,0.4812,45.1163,63.3962,36.5591,40.8377,42.1053
4,2022,No_Volatility,720,49.4444,49.7441,0.4992,39.3305,60.0917,49.8099,44.5702,40.2390
5,2021-2022,No_Volatility,1479,48.8844,48.9881,0.4912,42.0705,61.9048,42.9889,42.5297,41.1647



=== CHANGE AFTER REMOVING VOLATILITY ===


,Accuracy,Balanced_Accuracy,Macro_F1,Down_Recall,Up_Recall,Directional_Recall,Opposite_Error
Period,,,,,,,
2021,-0.9223,-0.9952,-0.0096,-2.3256,-1.7921,-2.0588,2.4291
2021-2022,1.1494,1.1201,0.0123,1.1013,1.8450,1.4732,0.0000
2022,3.3333,3.1429,0.0339,4.1841,5.7034,4.9438,-2.3904


**Singapore B: Bank-level stability check**

In [25]:

# Identify the bank or stock identifier in the validation data.
possible_names = {
    "ric", "riccode", "ticker", "symbol", "bank",
    "stock", "stockcode", "stockid", "stockname",
    "company", "companyname", "instrument",
    "instrumentric", "security", "securitycode"
}

bank_col = next(
    (
        col for col in valid.columns
        if str(col).lower().replace("_", "").replace(" ", "")
        in possible_names
    ),
    None
)

if bank_col is None:
    raise KeyError(
        f"Bank identifier not found. Available columns: "
        f"{list(valid.columns)}"
    )

bank_ids = valid[bank_col].astype(str).to_numpy()

bank_predictions = {}

# Refit the exact two Singapore B models from the yearly comparison.
for feature_set_name, selected_features in feature_sets.items():

    model = XGBClassifier(**model_params)

    model.fit(
        train[selected_features],
        y_train,
        sample_weight=sample_weights,
        verbose=False
    )

    bank_predictions[feature_set_name] = (
        model.predict(valid[selected_features]).astype(int)
    )

bank_results = []

for bank in sorted(np.unique(bank_ids)):

    for period in ["2021", "2022", "2021-2022"]:

        mask = bank_ids == bank

        if period != "2021-2022":
            mask &= validation_years == int(period)

        actual = y_valid[mask]

        if len(actual) == 0:
            continue

        metrics = {}

        for feature_set_name, predictions in bank_predictions.items():

            predicted = predictions[mask]

            recalls = recall_score(
                actual,
                predicted,
                labels=[0, 1, 2],
                average=None,
                zero_division=0
            )

            opposite = (
                ((actual == 0) & (predicted == 2))
                | ((actual == 2) & (predicted == 0))
            )

            direction_count = np.isin(actual, [0, 2]).sum()

            metrics[feature_set_name] = {
                "Macro_F1": f1_score(
                    actual, predicted,
                    labels=[0, 1, 2],
                    average="macro",
                    zero_division=0
                ),
                "Down_Recall": recalls[0] * 100,
                "Up_Recall": recalls[2] * 100,
                "Opposite_Error": (
                    opposite.sum() / direction_count * 100
                    if direction_count else np.nan
                )
            }

        full = metrics["All_23"]
        reduced = metrics["No_Volatility"]

        bank_results.append({
            "Bank": bank,
            "Period": period,
            "Rows": len(actual),
            "Macro_F1_23": full["Macro_F1"],
            "Macro_F1_20": reduced["Macro_F1"],
            "F1_Change":
                reduced["Macro_F1"] - full["Macro_F1"],
            "Down_Recall_Change":
                reduced["Down_Recall"] - full["Down_Recall"],
            "Up_Recall_Change":
                reduced["Up_Recall"] - full["Up_Recall"],
            "Opposite_Error_Change":
                reduced["Opposite_Error"] - full["Opposite_Error"]
        })

bank_stability_df = pd.DataFrame(bank_results)

display(bank_stability_df.round(4))

bank_stability_df.to_csv(
    output_dir / "sg_b_bank_level_ablation.csv",
    index=False
)


,Bank,Period,Rows,Macro_F1_23,Macro_F1_20,F1_Change,Down_Recall_Change,Up_Recall_Change,Opposite_Error_Change
0,D05.SI,2021,253,0.4897,0.4897,0.0000,-1.4085,0.0000,1.7857
1,D05.SI,2022,240,0.4463,0.4845,0.0382,3.6585,4.7619,-1.8072
2,D05.SI,2021-2022,493,0.4685,0.4872,0.0187,1.3072,2.2099,0.0000
3,O39.SI,2021,253,0.4742,0.4768,0.0026,-1.3514,-1.0638,1.1905
4,O39.SI,2022,240,0.5221,0.5460,0.0239,7.7922,3.0928,-1.7241
5,O39.SI,2021-2022,493,0.4981,0.5118,0.0137,3.3113,1.0471,-0.2924
6,U11.SI,2021,253,0.5069,0.4749,-0.0320,-4.2857,-4.5455,4.4304
7,U11.SI,2022,240,0.4273,0.4664,0.0391,1.2500,9.7561,-3.7037
8,U11.SI,2021-2022,493,0.4698,0.4743,0.0045,-1.3333,2.3529,0.3125


**Check data structure for window optimisation**

In [26]:

selected_configs = {
    "Malaysia": {
        "Approach": "C_Market_Specific",
        "Alpha": 0.5,
        "Horizon": 8
    },
    "Singapore": {
        "Approach": "B_Common",
        "Alpha": 1.0,
        "Horizon": 2
    }
}

# Identify date, stock identifier and label maturity columns.
all_columns = list(candidate_data.columns)

date_candidates = [
    col for col in all_columns
    if str(col).lower() in [
        "date", "trading_date", "tradingdate",
        "datetime", "timestamp"
    ]
]

bank_candidates = [
    col for col in all_columns
    if str(col).lower().replace("_", "") in [
        "ric", "ticker", "symbol", "bank",
        "stock", "stockid", "stockcode",
        "instrument", "instrumentric"
    ]
]

maturity_candidates = [
    col for col in all_columns
    if any(
        term in str(col).lower()
        for term in ["label_end", "event_end", "vertical_barrier", "t1"]
    )
]

print("Date columns:", date_candidates)
print("Bank identifier columns:", bank_candidates)
print("Label maturity columns:", maturity_candidates)
print("Datetime index:", isinstance(candidate_data.index, pd.DatetimeIndex))
print("Feature count:", len(feature_columns))
print("All columns:", all_columns)

# Check the available data for the two selected configurations.
data_checks = []

for country, config in selected_configs.items():

    subset = candidate_data.loc[
        (candidate_data["Country"] == country)
        & (candidate_data["Approach"] == config["Approach"])
    ]

    for split, group in subset.groupby("Split"):

        data_checks.append({
            "Country": country,
            "Approach": config["Approach"],
            "Split": split,
            "Rows": len(group),
            "Banks": (
                group[bank_candidates[0]].nunique()
                if bank_candidates else "Check identifier"
            ),
            "Date_Min": (
                str(group[date_candidates[0]].min())
                if date_candidates else "Check index"
            ),
            "Date_Max": (
                str(group[date_candidates[0]].max())
                if date_candidates else "Check index"
            )
        })

display(pd.DataFrame(data_checks))



Date columns: ['Date']
Bank identifier columns: ['Ticker']
Label maturity columns: ['Event_End_Date']
Datetime index: False
Feature count: 23
All columns: ['Ticker', 'Country', 'Date', 'future_date_10', 'returns', 'returns_lag_1', 'returns_lag_2', 'momentum_5', 'momentum_10', 'daily_range', 'candle_body', 'sma_5_ratio', 'sma_10_ratio', 'sma_20_ratio', 'volatility_5', 'volatility_10', 'volatility_20', 'rsi', 'macd', 'macd_signal', 'macd_diff', 'volume_change', 'volume_ratio_20', 'bb_high', 'bb_low', 'bb_width', 'skewness_20', 'Split', 'Event_End_Date', 'Label', 'Approach', 'V', 'H', 'M']


,Country,Approach,Split,Rows,Banks,Date_Min,Date_Max
0,Malaysia,C_Market_Specific,Train,13192,10,2015-02-24 00:00:00,2020-12-16 00:00:00
1,Malaysia,C_Market_Specific,Validation,4493,10,2021-01-04 00:00:00,2022-12-15 00:00:00
2,Singapore,B_Common,Train,4373,3,2015-02-18 00:00:00,2020-12-16 00:00:00
3,Singapore,B_Common,Validation,1479,3,2021-01-04 00:00:00,2022-12-15 00:00:00


**Training Window Comparison**

In [27]:

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    recall_score
)

window_candidates = [
    ("Malaysia", "C_Market_Specific", 0.5),
    ("Singapore", "B_Common", 1.0)
]

rolling_weeks = list(range(26, 313, 26))

window_yearly_results = []
window_overall_results = []

for country, approach, alpha in window_candidates:

    data = candidate_data.loc[
        (candidate_data["Country"] == country)
        & (candidate_data["Approach"] == approach)
        & (candidate_data["Split"].isin(["Train", "Validation"]))
    ].copy()

    data["Date"] = pd.to_datetime(data["Date"])
    data["Event_End_Date"] = pd.to_datetime(
        data["Event_End_Date"]
    )

    validation = data.loc[
        (data["Split"] == "Validation")
        & (data["Date"].dt.year.isin([2021, 2022]))
    ].sort_values(["Date", "Ticker"])

    validation_by_year = {
        year: validation.loc[
            validation["Date"].dt.year == year
        ]
        for year in [2021, 2022]
    }

    assert all(
        len(group) > 0
        for group in validation_by_year.values()
    )

    # The fixed window must remain unchanged across validation years.
    first_cutoff = validation_by_year[2021]["Date"].min()

    strategies = (
        [("Fixed", None), ("Expanding", None)]
        + [
            (f"Rolling_{weeks}w", weeks)
            for weeks in rolling_weeks
        ]
    )

    for strategy, weeks in strategies:

        actual_all = []
        predicted_all = []
        train_sizes = {}

        yearly_scores = {}

        for year in [2021, 2022]:

            test = validation_by_year[year]
            cutoff = test["Date"].min()

            # Fixed models retain only the original training period.
            if strategy == "Fixed":
                train = data.loc[
                    (data["Split"] == "Train")
                    & (data["Date"] < first_cutoff)
                    & (data["Event_End_Date"] < first_cutoff)
                ]

            else:
                # A label may be used only after its event has ended.
                train = data.loc[
                    (data["Date"] < cutoff)
                    & (data["Event_End_Date"] < cutoff)
                ]

                if weeks is not None:
                    window_start = cutoff - pd.Timedelta(
                        weeks=weeks
                    )

                    train = train.loc[
                        train["Date"] >= window_start
                    ]

            train = train.sort_values(["Date", "Ticker"])

            assert len(train) > 0

            X_train = train[feature_columns]
            y_train = (
                train["Label"].to_numpy(dtype=int) + 1
            )

            X_test = test[feature_columns]
            y_test = (
                test["Label"].to_numpy(dtype=int) + 1
            )

            counts = np.bincount(
                y_train, minlength=3
            )

            assert (counts > 0).all(), (
                country, strategy, year, counts
            )

            # Recalculate class weights using this training window only.
            balanced_weights = (
                len(y_train) / (3.0 * counts)
            )

            sample_weights = (
                balanced_weights[y_train] ** alpha
            )

            sample_weights /= sample_weights.mean()

            model = XGBClassifier(**model_params)

            model.fit(
                X_train,
                y_train,
                sample_weight=sample_weights,
                verbose=False
            )

            predictions = model.predict(
                X_test
            ).astype(int)

            recalls = recall_score(
                y_test,
                predictions,
                labels=[0, 1, 2],
                average=None,
                zero_division=0
            )

            directional_mask = np.isin(
                y_test, [0, 2]
            )

            opposite_errors = (
                ((y_test == 0) & (predictions == 2))
                | ((y_test == 2) & (predictions == 0))
            )

            yearly_f1 = f1_score(
                y_test,
                predictions,
                average="macro",
                zero_division=0
            )

            train_sizes[year] = len(train)
            yearly_scores[year] = yearly_f1

            window_yearly_results.append({
                "Country": country,
                "Approach": approach,
                "Alpha": alpha,
                "Strategy": strategy,
                "Year": year,
                "Train_Rows": len(train),
                "Test_Rows": len(test),
                "Accuracy": accuracy_score(
                    y_test, predictions
                ) * 100,
                "Balanced_Accuracy":
                    balanced_accuracy_score(
                        y_test, predictions
                    ) * 100,
                "Macro_F1": yearly_f1,
                "Down_Recall": recalls[0] * 100,
                "Flat_Recall": recalls[1] * 100,
                "Up_Recall": recalls[2] * 100,
                "Directional_Recall":
                    (recalls[0] + recalls[2]) * 50,
                "Opposite_Error":
                    opposite_errors.sum()
                    / directional_mask.sum() * 100
            })

            actual_all.append(y_test)
            predicted_all.append(predictions)

        # Pool the predictions from both years rather than
        # averaging the two yearly performance scores.
        actual = np.concatenate(actual_all)
        predicted = np.concatenate(predicted_all)

        recalls = recall_score(
            actual,
            predicted,
            labels=[0, 1, 2],
            average=None,
            zero_division=0
        )

        directional_mask = np.isin(actual, [0, 2])

        opposite_errors = (
            ((actual == 0) & (predicted == 2))
            | ((actual == 2) & (predicted == 0))
        )

        window_overall_results.append({
            "Country": country,
            "Approach": approach,
            "Alpha": alpha,
            "Strategy": strategy,
            "Train_2021": train_sizes[2021],
            "Train_2022": train_sizes[2022],
            "F1_2021": yearly_scores[2021],
            "F1_2022": yearly_scores[2022],
            "Accuracy": accuracy_score(
                actual, predicted
            ) * 100,
            "Balanced_Accuracy":
                balanced_accuracy_score(
                    actual, predicted
                ) * 100,
            "Macro_F1": f1_score(
                actual,
                predicted,
                average="macro",
                zero_division=0
            ),
            "Down_Recall": recalls[0] * 100,
            "Flat_Recall": recalls[1] * 100,
            "Up_Recall": recalls[2] * 100,
            "Directional_Recall":
                (recalls[0] + recalls[2]) * 50,
            "Opposite_Error":
                opposite_errors.sum()
                / directional_mask.sum() * 100
        })

window_overall_df = pd.DataFrame(
    window_overall_results
)

window_yearly_df = pd.DataFrame(
    window_yearly_results
)

for country in ["Malaysia", "Singapore"]:

    print(
        f"\n=== {country.upper()} — "
        "TRAINING WINDOW COMPARISON (2021–2022) ==="
    )

    results = window_overall_df.loc[
        window_overall_df["Country"] == country
    ]

    display(
        results[
            [
                "Strategy",
                "Train_2021",
                "Train_2022",
                "F1_2021",
                "F1_2022",
                "Accuracy",
                "Balanced_Accuracy",
                "Macro_F1",
                "Down_Recall",
                "Flat_Recall",
                "Up_Recall",
                "Directional_Recall",
                "Opposite_Error"
            ]
        ].round(4)
    )

window_overall_df.to_csv(
    output_dir / "selected_window_comparison.csv",
    index=False
)

window_yearly_df.to_csv(
    output_dir / "selected_window_yearly.csv",
    index=False
)



=== MALAYSIA — TRAINING WINDOW COMPARISON (2021–2022) ===


,Strategy,Train_2021,Train_2022,F1_2021,F1_2022,Accuracy,Balanced_Accuracy,Macro_F1,Down_Recall,Flat_Recall,Up_Recall,Directional_Recall,Opposite_Error
0,Fixed,13192,13192,0.5132,0.5321,52.9045,53.0165,0.5235,54.8978,72.1569,31.9948,43.4463,42.6595
1,Expanding,13192,15384,0.5132,0.5331,53.0381,53.1900,0.5236,56.5187,72.2222,30.8290,43.6738,42.4232
2,Rolling_26w,1088,1094,0.4871,0.5005,49.6105,49.8207,0.4948,56.0958,61.2418,32.1244,44.1101,42.0182
3,Rolling_52w,2292,2192,0.4887,0.5144,50.0556,50.2849,0.5021,57.3643,60.6536,32.8368,45.1006,44.1782
4,Rolling_78w,3469,3280,0.5014,0.5056,50.4118,50.6130,0.5037,56.3777,63.7255,31.7358,44.0567,44.2119
5,Rolling_104w,4626,4484,0.5162,0.5005,51.2575,51.5180,0.5084,59.3376,65.2941,29.9223,44.6299,42.5920
6,Rolling_130w,5786,5661,0.5099,0.4970,51.1240,51.3637,0.5037,58.0691,68.4314,27.5907,42.8299,43.2332
7,Rolling_156w,6874,6818,0.5091,0.5159,51.9920,52.2460,0.5126,59.5490,68.5621,28.6269,44.0880,42.3220
8,Rolling_182w,7759,7978,0.5027,0.5121,51.5914,51.8398,0.5075,58.7738,69.6732,27.0725,42.9232,43.5370
9,Rolling_208w,8830,9066,0.5097,0.5142,51.9252,52.1574,0.5120,58.5624,69.4771,28.4326,43.4975,43.0307



=== SINGAPORE — TRAINING WINDOW COMPARISON (2021–2022) ===


,Strategy,Train_2021,Train_2022,F1_2021,F1_2022,Accuracy,Balanced_Accuracy,Macro_F1,Down_Recall,Flat_Recall,Up_Recall,Directional_Recall,Opposite_Error
14,Fixed,4373,4373,0.4954,0.4677,48.2082,48.2106,0.4823,39.2070,61.6977,43.7269,41.4670,40.1606
15,Expanding,4373,5126,0.4954,0.4719,48.4787,48.4685,0.4846,39.4273,61.6977,44.2804,41.8539,39.2570
16,Rolling_26w,343,375,0.4031,0.3886,39.5538,40.1910,0.3968,46.4758,46.7909,27.3063,36.8910,42.4699
17,Rolling_52w,718,753,0.4195,0.4138,41.5145,41.9802,0.4183,47.1366,45.9627,32.8413,39.9889,41.7671
18,Rolling_78w,1092,1096,0.4771,0.4546,46.3151,46.3762,0.4662,41.8502,54.6584,42.6199,42.2351,40.4618
19,Rolling_104w,1462,1471,0.4372,0.4537,44.4895,44.8517,0.4479,43.3921,56.1077,35.0554,39.2237,42.4699
20,Rolling_130w,1837,1845,0.4231,0.4539,44.0162,44.4416,0.4428,41.4097,59.6273,32.2878,36.8488,45.2811
21,Rolling_156w,2212,2215,0.4460,0.4663,45.5713,46.0005,0.4587,44.4934,59.0062,34.5018,39.4976,43.1727
22,Rolling_182w,2587,2590,0.4527,0.4400,44.4895,44.7034,0.4465,38.9868,58.5921,36.5314,37.7591,42.7711
23,Rolling_208w,2962,2965,0.5006,0.4747,48.6815,48.7346,0.4882,41.6300,60.6625,43.9114,42.7707,39.5582


**XGBoost hyperparameter tuning**

In [29]:

from pathlib import Path

import numpy as np
import pandas as pd
import optuna

from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    recall_score
)

optuna.logging.set_verbosity(optuna.logging.WARNING)

output_dir = Path(output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

tuning_specs = [
    ("Malaysia", "C_Market_Specific", 0.5, "Expanding"),
    ("Singapore", "B_Common", 1.0, "Rolling_208w")
]

trials_per_market = 100

search_parameters = [
    "n_estimators",
    "learning_rate",
    "max_depth",
    "min_child_weight",
    "gamma",
    "subsample",
    "colsample_bytree",
    "reg_alpha",
    "reg_lambda"
]


def prepare_tuning_folds(country, approach, alpha, strategy):

    data = candidate_data.loc[
        (candidate_data["Country"] == country)
        & (candidate_data["Approach"] == approach)
        & (candidate_data["Split"].isin(["Train", "Validation"]))
    ].copy()

    data["Date"] = pd.to_datetime(data["Date"])
    data["Event_End_Date"] = pd.to_datetime(
        data["Event_End_Date"]
    )

    folds = []

    for year in [2021, 2022]:

        test = data.loc[
            (data["Split"] == "Validation")
            & (data["Date"].dt.year == year)
        ].sort_values(["Date", "Ticker"])

        cutoff = test["Date"].min()

        # Training labels must mature before the prediction cutoff.
        train = data.loc[
            (data["Date"] < cutoff)
            & (data["Event_End_Date"] < cutoff)
        ]

        if strategy == "Rolling_208w":
            train = train.loc[
                train["Date"] >= (
                    cutoff - pd.Timedelta(weeks=208)
                )
            ]

        train = train.sort_values(["Date", "Ticker"])

        assert len(train) > 0 and len(test) > 0
        assert train["Event_End_Date"].max() < cutoff

        y_train = train["Label"].to_numpy(dtype=int) + 1
        y_test = test["Label"].to_numpy(dtype=int) + 1

        counts = np.bincount(y_train, minlength=3)
        assert (counts > 0).all()

        # Class weights are computed separately for each training fold.
        balanced_weights = len(y_train) / (3.0 * counts)
        sample_weights = balanced_weights[y_train] ** alpha
        sample_weights /= sample_weights.mean()

        folds.append({
            "Year": year,
            "X_train": train[feature_columns],
            "y_train": y_train,
            "weights": sample_weights,
            "X_test": test[feature_columns],
            "y_test": y_test
        })

    return folds


def evaluate_tuning_params(params, folds):

    actual_all = []
    predicted_all = []
    yearly_f1 = {}

    for fold in folds:

        model = XGBClassifier(**params)

        model.fit(
            fold["X_train"],
            fold["y_train"],
            sample_weight=fold["weights"],
            verbose=False
        )

        actual = fold["y_test"]
        predicted = model.predict(
            fold["X_test"]
        ).astype(int)

        yearly_f1[fold["Year"]] = f1_score(
            actual,
            predicted,
            average="macro",
            zero_division=0
        )

        actual_all.append(actual)
        predicted_all.append(predicted)

    actual = np.concatenate(actual_all)
    predicted = np.concatenate(predicted_all)

    # Class order: 0 = Down, 1 = Flat, 2 = Up.
    recalls = recall_score(
        actual,
        predicted,
        labels=[0, 1, 2],
        average=None,
        zero_division=0
    )

    directional_actual = np.isin(actual, [0, 2])
    directional_predicted = np.isin(predicted, [0, 2])

    correct_signals = (
        directional_predicted & (actual == predicted)
    )

    opposite_errors = (
        ((actual == 0) & (predicted == 2))
        | ((actual == 2) & (predicted == 0))
    )

    signal_precision = (
        correct_signals.sum()
        / directional_predicted.sum() * 100
        if directional_predicted.any()
        else 0.0
    )

    opposite_rate = (
        opposite_errors.sum()
        / directional_actual.sum() * 100
        if directional_actual.any()
        else 0.0
    )

    return {
        "F1_2021": yearly_f1[2021],
        "F1_2022": yearly_f1[2022],
        "Min_Year_F1": min(yearly_f1.values()),
        "Accuracy": accuracy_score(
            actual, predicted
        ) * 100,
        "Balanced_Accuracy": balanced_accuracy_score(
            actual, predicted
        ) * 100,
        "Macro_F1": f1_score(
            actual,
            predicted,
            average="macro",
            zero_division=0
        ),
        "Down_Recall": recalls[0] * 100,
        "Flat_Recall": recalls[1] * 100,
        "Up_Recall": recalls[2] * 100,
        "Directional_Recall":
            (recalls[0] + recalls[2]) * 50,
        "Signal_Precision": signal_precision,
        "Opposite_Error": opposite_rate
    }


def suggest_xgb_params(trial):

    # Broad but bounded ranges for model capacity and regularisation.
    return {
        "n_estimators": trial.suggest_int(
            "n_estimators", 200, 1500, step=50
        ),
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.01, 0.20, log=True
        ),
        "max_depth": trial.suggest_int(
            "max_depth", 2, 10
        ),
        "min_child_weight": trial.suggest_int(
            "min_child_weight", 1, 20
        ),
        "gamma": trial.suggest_float(
            "gamma", 0.0, 5.0
        ),
        "subsample": trial.suggest_float(
            "subsample", 0.50, 1.00
        ),
        "colsample_bytree": trial.suggest_float(
            "colsample_bytree", 0.50, 1.00
        ),
        "reg_alpha": trial.suggest_float(
            "reg_alpha", 1e-8, 10.0, log=True
        ),
        "reg_lambda": trial.suggest_float(
            "reg_lambda", 1e-3, 100.0, log=True
        )
    }


def tuning_objective(trial, folds):

    trial_params = suggest_xgb_params(trial)

    # Preserve existing XGBoost settings not being tuned.
    params = {
        **model_params,
        **trial_params
    }

    metrics = evaluate_tuning_params(params, folds)

    # Save every evaluation metric, not only the objective score.
    for metric, value in metrics.items():
        trial.set_user_attr(metric, float(value))

    return metrics["Macro_F1"]


tuning_all_results = []
tuning_best_params = {}

for country, approach, alpha, strategy in tuning_specs:

    folds = prepare_tuning_folds(
        country, approach, alpha, strategy
    )

    # Evaluate the original, untuned XGBoost model.
    baseline_metrics = evaluate_tuning_params(
        model_params, folds
    )

    # Confirm consistency with the earlier window experiment.
    previous_f1 = window_overall_df.loc[
        (window_overall_df["Country"] == country)
        & (window_overall_df["Strategy"] == strategy),
        "Macro_F1"
    ].iloc[0]

    assert np.isclose(
        baseline_metrics["Macro_F1"],
        previous_f1,
        atol=1e-8
    ), f"Baseline mismatch: {country}"

    # SQLite storage preserves trials if the notebook is interrupted.
    db_path = (
        output_dir / f"nb06_tpe100_{country.lower()}_v1.db"
    ).resolve()

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(
            seed=42,
            n_startup_trials=20
        ),
        study_name=f"nb06_{country.lower()}_tpe100_v1",
        storage=f"sqlite:///{db_path.as_posix()}",
        load_if_exists=True
    )

    # Resume an existing study rather than repeating completed trials.
    completed_trials = sum(
        t.state == optuna.trial.TrialState.COMPLETE
        for t in study.trials
    )

    remaining_trials = max(
        0, trials_per_market - completed_trials
    )

    if remaining_trials > 0:
        study.optimize(
            lambda trial: tuning_objective(trial, folds),
            n_trials=remaining_trials,
            show_progress_bar=True
        )

    # Baseline is displayed separately from the Optuna trial budget.
    baseline_row = {
        "Country": country,
        "Strategy": strategy,
        "Trial": 0,
        "Type": "Baseline",
        **baseline_metrics
    }

    for parameter in search_parameters:
        baseline_row[f"param_{parameter}"] = (
            model_params.get(parameter, "XGBoost default")
        )

    country_rows = [baseline_row]

    for trial in study.trials:

        if trial.state != optuna.trial.TrialState.COMPLETE:
            continue

        row = {
            "Country": country,
            "Strategy": strategy,
            "Trial": trial.number + 1,
            "Type": "Optuna_TPE",
            **trial.user_attrs
        }

        for parameter in search_parameters:
            row[f"param_{parameter}"] = trial.params.get(
                parameter
            )

        country_rows.append(row)

    country_results = pd.DataFrame(country_rows)

    # Select top trials by pooled Macro F1.
    # Minimum yearly F1 is used only as a tie-breaker.
    tuned_results = country_results.loc[
        country_results["Type"] == "Optuna_TPE"
    ].sort_values(
        ["Macro_F1", "Min_Year_F1"],
        ascending=False
    )

    top_five = tuned_results.head(5)

    comparison = pd.concat([
        country_results.loc[
            country_results["Type"] == "Baseline"
        ],
        top_five
    ])

    performance_columns = [
        "Trial",
        "Type",
        "F1_2021",
        "F1_2022",
        "Accuracy",
        "Balanced_Accuracy",
        "Macro_F1",
        "Down_Recall",
        "Flat_Recall",
        "Up_Recall",
        "Directional_Recall",
        "Signal_Precision",
        "Opposite_Error"
    ]

    print(
        f"\n=== {country.upper()} — "
        "BASELINE VS TOP 5 TUNED MODELS ==="
    )

    display(
        comparison[performance_columns]
        .round(4)
        .reset_index(drop=True)
    )

    # Display actual hyperparameters for the same six models.
    parameter_columns = [
        "Trial",
        "Type"
    ] + [
        f"param_{parameter}"
        for parameter in search_parameters
    ]

    print(
        f"\n=== {country.upper()} — "
        "HYPERPARAMETERS FOR TOP MODELS ==="
    )

    display(
        comparison[parameter_columns]
        .round(6)
        .reset_index(drop=True)
    )

    # Include the baseline when identifying the strongest candidate.
    best_overall = country_results.sort_values(
        ["Macro_F1", "Min_Year_F1"],
        ascending=False
    ).iloc[0]

    best_trial = int(best_overall["Trial"])

    if best_trial == 0:
        best_params = dict(model_params)
    else:
        best_trial_object = study.trials[best_trial - 1]
        best_params = {
            **model_params,
            **best_trial_object.params
        }

    tuning_best_params[country] = best_params

    print(
        f"\n{country} — Best candidate trial: {best_trial}"
    )
    print(
        f"Macro F1: {best_overall['Macro_F1']:.4f}"
    )
    print("Parameters:", best_params)

    # Save all completed trials and their hyperparameter settings.
    country_results.to_csv(
        output_dir / (
            f"xgb_optuna_tuning_{country.lower()}.csv"
        ),
        index=False
    )

    tuning_all_results.extend(country_rows)

tuning_results_df = pd.DataFrame(tuning_all_results)

tuning_results_df.to_csv(
    output_dir / "xgb_optuna_tuning_all_results.csv",
    index=False
)


  0%|          | 0/100 [00:00<?, ?it/s]


=== MALAYSIA — BASELINE VS TOP 5 TUNED MODELS ===


,Trial,Type,F1_2021,F1_2022,Accuracy,Balanced_Accuracy,Macro_F1,Down_Recall,Flat_Recall,Up_Recall,Directional_Recall,Signal_Precision,Opposite_Error
0,0,Baseline,0.5132,0.5331,53.0381,53.1900,0.5236,56.5187,72.2222,30.8290,43.6738,43.1757,42.4232
1,86,Optuna_TPE,0.5361,0.5422,54.6628,54.6086,0.5395,50.2467,76.1438,37.4352,43.8409,45.7153,39.4195
2,71,Optuna_TPE,0.5369,0.5378,54.3512,54.3056,0.5375,50.3876,74.6405,37.8886,44.1381,45.5661,39.3183
3,64,Optuna_TPE,0.5308,0.5432,54.3512,54.2955,0.5374,49.9648,75.0980,37.8238,43.8943,45.3684,39.6895
4,78,Optuna_TPE,0.5350,0.5380,54.2844,54.2382,0.5366,50.2467,75.0327,37.4352,43.8409,45.2506,39.8245
5,81,Optuna_TPE,0.5357,0.5370,54.2399,54.1636,0.5366,49.1191,74.7059,38.6658,43.8925,45.3876,39.4870



=== MALAYSIA — HYPERPARAMETERS FOR TOP MODELS ===


,Trial,Type,param_n_estimators,param_learning_rate,param_max_depth,param_min_child_weight,param_gamma,param_subsample,param_colsample_bytree,param_reg_alpha,param_reg_lambda
0,0,Baseline,200,0.050000,3,3,XGBoost default,0.800000,0.800000,XGBoost default,XGBoost default
1,86,Optuna_TPE,300,0.017625,10,11,0.790205,0.568075,0.962902,0.002925,0.003498
2,71,Optuna_TPE,350,0.019147,9,10,0.733119,0.533144,0.981575,0.010841,0.024834
3,64,Optuna_TPE,300,0.022184,9,11,1.174169,0.532943,0.991082,0.0023,0.007868
4,78,Optuna_TPE,450,0.015127,10,10,0.796307,0.598160,0.978555,0.0027,0.023229
5,81,Optuna_TPE,450,0.023272,9,11,0.593307,0.550800,0.989841,0.002576,0.004034



Malaysia — Best candidate trial: 86
Macro F1: 0.5395
Parameters: {'n_estimators': 300, 'learning_rate': 0.017624958652519118, 'max_depth': 10, 'min_child_weight': 11, 'subsample': 0.5680753394556178, 'colsample_bytree': 0.9629020587630486, 'objective': 'multi:softprob', 'num_class': 3, 'eval_metric': 'mlogloss', 'tree_method': 'hist', 'random_state': 42, 'n_jobs': -1, 'gamma': 0.7902045732098001, 'reg_alpha': 0.0029247230310254597, 'reg_lambda': 0.0034981946346891255}


... (2 outputs left out)

,Trial,Type,F1_2021,F1_2022,Accuracy,Balanced_Accuracy,Macro_F1,Down_Recall,Flat_Recall,Up_Recall,Directional_Recall,Signal_Precision,Opposite_Error
0,0,Baseline,0.5006,0.4747,48.6815,48.7346,0.4882,41.6300,60.6625,43.9114,42.7707,42.2354,39.5582
1,28,Optuna_TPE,0.4901,0.4891,48.9520,49.1092,0.4899,43.1718,61.9048,42.2509,42.7114,42.9727,38.1526
2,50,Optuna_TPE,0.5037,0.4711,48.7492,48.8205,0.4881,40.9692,62.3188,43.1734,42.0713,42.0842,39.7590
3,38,Optuna_TPE,0.4961,0.4775,48.6815,48.8310,0.4868,41.1894,63.9752,41.3284,41.2589,41.8108,39.9598
4,76,Optuna_TPE,0.4952,0.4767,48.6139,48.6710,0.4860,39.6476,63.5611,42.8044,41.2260,41.6582,40.2610
5,91,Optuna_TPE,0.4907,0.4804,48.5463,48.6974,0.4857,41.4097,63.3540,41.3284,41.3691,41.7427,39.9598



=== SINGAPORE — HYPERPARAMETERS FOR TOP MODELS ===


,Trial,Type,param_n_estimators,param_learning_rate,param_max_depth,param_min_child_weight,param_gamma,param_subsample,param_colsample_bytree,param_reg_alpha,param_reg_lambda
0,0,Baseline,200,0.050000,3,3,XGBoost default,0.800000,0.800000,XGBoost default,XGBoost default
1,28,Optuna_TPE,550,0.023619,6,1,4.304066,0.524898,0.947631,0.05582,0.015142
2,50,Optuna_TPE,500,0.078754,6,1,3.970445,0.515401,0.903726,0.031982,0.011235
3,38,Optuna_TPE,450,0.025148,6,3,4.187225,0.586909,0.891594,0.01841,0.001664
4,76,Optuna_TPE,650,0.019058,7,1,3.829214,0.575485,0.988824,0.240217,0.014099
5,91,Optuna_TPE,500,0.025403,7,1,3.683276,0.525790,0.912823,0.200822,0.003645



Singapore — Best candidate trial: 28
Macro F1: 0.4899
Parameters: {'n_estimators': 550, 'learning_rate': 0.02361897070338257, 'max_depth': 6, 'min_child_weight': 1, 'subsample': 0.5248982220765283, 'colsample_bytree': 0.9476309179343638, 'objective': 'multi:softprob', 'num_class': 3, 'eval_metric': 'mlogloss', 'tree_method': 'hist', 'random_state': 42, 'n_jobs': -1, 'gamma': 4.3040661723546, 'reg_alpha': 0.05581964584664106, 'reg_lambda': 0.015142477564097125}


**Recheck Singapore's feature selection after tuning**

In [30]:

sg_folds = prepare_tuning_folds(
    "Singapore", "B_Common", 1.0, "Rolling_208w"
)

volatility_features = [
    "volatility_5",
    "volatility_10",
    "volatility_20"
]

sg_feature_sets = {
    "All_23": list(feature_columns),
    "Without_Volatility_20": [
        f for f in feature_columns
        if f not in volatility_features
    ]
}

assert len(sg_feature_sets["Without_Volatility_20"]) == 20

sg_parameter_sets = {
    "Baseline": dict(model_params),
    "Tuned_28": dict(tuning_best_params["Singapore"])
}


def calculate_sg_metrics(actual, predicted):

    recalls = recall_score(
        actual,
        predicted,
        labels=[0, 1, 2],
        average=None,
        zero_division=0
    )

    directional_actual = np.isin(actual, [0, 2])
    directional_predicted = np.isin(predicted, [0, 2])

    correct_signals = (
        directional_predicted & (actual == predicted)
    )

    opposite_errors = (
        ((actual == 0) & (predicted == 2))
        | ((actual == 2) & (predicted == 0))
    )

    return {
        "Accuracy": accuracy_score(
            actual, predicted
        ) * 100,
        "Balanced_Accuracy": balanced_accuracy_score(
            actual, predicted
        ) * 100,
        "Macro_F1": f1_score(
            actual, predicted,
            average="macro",
            zero_division=0
        ),
        "Down_Recall": recalls[0] * 100,
        "Flat_Recall": recalls[1] * 100,
        "Up_Recall": recalls[2] * 100,
        "Directional_Recall":
            (recalls[0] + recalls[2]) * 50,
        "Signal_Precision": (
            correct_signals.sum()
            / directional_predicted.sum() * 100
            if directional_predicted.any()
            else 0.0
        ),
        "Opposite_Error": (
            opposite_errors.sum()
            / directional_actual.sum() * 100
            if directional_actual.any()
            else 0.0
        )
    }


sg_feature_recheck_results = []

for parameter_name, params in sg_parameter_sets.items():

    for feature_name, selected_features in sg_feature_sets.items():

        actual_all = []
        predicted_all = []

        for fold in sg_folds:

            model = XGBClassifier(**params)

            model.fit(
                fold["X_train"][selected_features],
                fold["y_train"],
                sample_weight=fold["weights"],
                verbose=False
            )

            actual = fold["y_test"]
            predicted = model.predict(
                fold["X_test"][selected_features]
            ).astype(int)

            metrics = calculate_sg_metrics(actual, predicted)

            sg_feature_recheck_results.append({
                "Parameters": parameter_name,
                "Feature_Set": feature_name,
                "Period": str(fold["Year"]),
                **metrics
            })

            actual_all.append(actual)
            predicted_all.append(predicted)

        # Calculate pooled metrics across both validation years.
        combined_metrics = calculate_sg_metrics(
            np.concatenate(actual_all),
            np.concatenate(predicted_all)
        )

        sg_feature_recheck_results.append({
            "Parameters": parameter_name,
            "Feature_Set": feature_name,
            "Period": "2021-2022",
            **combined_metrics
        })


sg_feature_recheck_df = pd.DataFrame(
    sg_feature_recheck_results
)

performance_columns = [
    "Parameters",
    "Feature_Set",
    "Accuracy",
    "Balanced_Accuracy",
    "Macro_F1",
    "Down_Recall",
    "Flat_Recall",
    "Up_Recall",
    "Directional_Recall",
    "Signal_Precision",
    "Opposite_Error"
]

# Verify that the 23-feature results match the tuning experiment.
expected_f1 = {
    "Baseline": 0.4882,
    "Tuned_28": 0.4899
}

for parameter_name, expected in expected_f1.items():

    actual_f1 = sg_feature_recheck_df.loc[
        (sg_feature_recheck_df["Parameters"] == parameter_name)
        & (sg_feature_recheck_df["Feature_Set"] == "All_23")
        & (sg_feature_recheck_df["Period"] == "2021-2022"),
        "Macro_F1"
    ].iloc[0]

    assert np.isclose(actual_f1, expected, atol=0.00005)


print("=== SINGAPORE — OVERALL FEATURE COMPARISON ===")

display(
    sg_feature_recheck_df.loc[
        sg_feature_recheck_df["Period"] == "2021-2022",
        performance_columns
    ].round(4).reset_index(drop=True)
)

print("\n=== SINGAPORE — YEARLY FEATURE COMPARISON ===")

display(
    sg_feature_recheck_df.loc[
        sg_feature_recheck_df["Period"] != "2021-2022",
        ["Period"] + performance_columns
    ].round(4).reset_index(drop=True)
)

sg_feature_recheck_df.to_csv(
    output_dir / "sg_feature_recheck_after_tuning.csv",
    index=False
)


=== SINGAPORE — OVERALL FEATURE COMPARISON ===


,Parameters,Feature_Set,Accuracy,Balanced_Accuracy,Macro_F1,Down_Recall,Flat_Recall,Up_Recall,Directional_Recall,Signal_Precision,Opposite_Error
0,Baseline,All_23,48.6815,48.7346,0.4882,41.6300,60.6625,43.9114,42.7707,42.2354,39.5582
1,Baseline,Without_Volatility_20,47.2617,47.3014,0.4750,39.4273,60.0414,42.4354,40.9314,39.9804,42.2691
2,Tuned_28,All_23,48.9520,49.1092,0.4899,43.1718,61.9048,42.2509,42.7114,42.9727,38.1526
3,Tuned_28,Without_Volatility_20,47.7350,47.9205,0.4781,40.9692,62.9400,39.8524,40.4108,40.5651,41.1647



=== SINGAPORE — YEARLY FEATURE COMPARISON ===


,Period,Parameters,Feature_Set,Accuracy,Balanced_Accuracy,Macro_F1,Down_Recall,Flat_Recall,Up_Recall,Directional_Recall,Signal_Precision,Opposite_Error
0,2021,Baseline,All_23,50.4611,49.9946,0.5006,41.8605,61.8868,46.2366,44.0485,43.6255,36.8421
1,2022,Baseline,All_23,46.8056,47.3473,0.4747,41.4226,59.1743,41.4449,41.4337,40.8644,42.2311
2,2021,Baseline,Without_Volatility_20,48.8801,48.4835,0.4860,40.9302,61.5094,43.0108,41.9705,40.7843,40.4858
3,2022,Baseline,Without_Volatility_20,45.5556,46.0524,0.4629,38.0753,58.2569,41.8251,39.9502,39.1813,44.0239
4,2021,Tuned_28,All_23,49.4071,49.1518,0.4901,43.2558,62.2642,41.9355,42.5956,42.5101,37.2470
5,2022,Tuned_28,All_23,48.4722,49.0499,0.4891,43.0962,61.4679,42.5856,42.8409,43.4343,39.0438
6,2021,Tuned_28,Without_Volatility_20,48.6166,48.3180,0.4823,41.3953,63.7736,39.7849,40.5901,40.0802,41.0931
7,2022,Tuned_28,Without_Volatility_20,46.8056,47.4788,0.4727,40.5858,61.9266,39.9240,40.2549,41.0569,41.2351


**Malaysia feature ablation after tuning**

In [31]:

my_folds = prepare_tuning_folds(
    "Malaysia", "C_Market_Specific", 0.5, "Expanding"
)

my_tuned_params = dict(tuning_best_params["Malaysia"])

# Use the same 10 feature groups from the earlier ablation.
assert set(
    feature
    for features in feature_groups.values()
    for feature in features
) == set(feature_columns)

my_ablation_sets = {
    "All_23": list(feature_columns)
}

for group_name, removed_features in feature_groups.items():
    my_ablation_sets[f"Remove_{group_name}"] = [
        feature for feature in feature_columns
        if feature not in removed_features
    ]


def calculate_my_ablation_metrics(actual, predicted):

    recalls = recall_score(
        actual,
        predicted,
        labels=[0, 1, 2],
        average=None,
        zero_division=0
    )

    directional_actual = np.isin(actual, [0, 2])
    directional_predicted = np.isin(predicted, [0, 2])

    correct_signals = (
        directional_predicted & (actual == predicted)
    )

    opposite_errors = (
        ((actual == 0) & (predicted == 2))
        | ((actual == 2) & (predicted == 0))
    )

    return {
        "Accuracy": accuracy_score(
            actual, predicted
        ) * 100,
        "Balanced_Accuracy": balanced_accuracy_score(
            actual, predicted
        ) * 100,
        "Macro_F1": f1_score(
            actual,
            predicted,
            average="macro",
            zero_division=0
        ),
        "Down_Recall": recalls[0] * 100,
        "Flat_Recall": recalls[1] * 100,
        "Up_Recall": recalls[2] * 100,
        "Directional_Recall":
            (recalls[0] + recalls[2]) * 50,
        "Signal_Precision": (
            correct_signals.sum()
            / directional_predicted.sum() * 100
            if directional_predicted.any() else 0.0
        ),
        "Opposite_Error": (
            opposite_errors.sum()
            / directional_actual.sum() * 100
            if directional_actual.any() else 0.0
        )
    }


my_tuned_ablation_results = []

for experiment, selected_features in my_ablation_sets.items():

    actual_all = []
    predicted_all = []

    for fold in my_folds:

        model = XGBClassifier(**my_tuned_params)

        model.fit(
            fold["X_train"][selected_features],
            fold["y_train"],
            sample_weight=fold["weights"],
            verbose=False
        )

        actual = fold["y_test"]
        predicted = model.predict(
            fold["X_test"][selected_features]
        ).astype(int)

        metrics = calculate_my_ablation_metrics(
            actual, predicted
        )

        my_tuned_ablation_results.append({
            "Experiment": experiment,
            "Features": len(selected_features),
            "Period": str(fold["Year"]),
            **metrics
        })

        actual_all.append(actual)
        predicted_all.append(predicted)

    # Pool both validation years using their actual predictions.
    combined_metrics = calculate_my_ablation_metrics(
        np.concatenate(actual_all),
        np.concatenate(predicted_all)
    )

    my_tuned_ablation_results.append({
        "Experiment": experiment,
        "Features": len(selected_features),
        "Period": "2021-2022",
        **combined_metrics
    })


my_tuned_ablation_df = pd.DataFrame(
    my_tuned_ablation_results
)

overall = my_tuned_ablation_df.loc[
    my_tuned_ablation_df["Period"] == "2021-2022"
].copy()

baseline_f1 = overall.loc[
    overall["Experiment"] == "All_23",
    "Macro_F1"
].iloc[0]

# Confirm the full-feature model reproduces tuned Trial 86.
assert np.isclose(
    baseline_f1, 0.5395, atol=0.00005
)

overall["F1_Change"] = (
    overall["Macro_F1"] - baseline_f1
)

print("=== MALAYSIA — TUNED FEATURE ABLATION (2021–2022) ===")

display(
    overall[
        [
            "Experiment",
            "Features",
            "Accuracy",
            "Balanced_Accuracy",
            "Macro_F1",
            "F1_Change",
            "Down_Recall",
            "Flat_Recall",
            "Up_Recall",
            "Directional_Recall",
            "Signal_Precision",
            "Opposite_Error"
        ]
    ].round(4).reset_index(drop=True)
)

# Compare each feature removal separately in 2021 and 2022.
yearly = my_tuned_ablation_df.loc[
    my_tuned_ablation_df["Period"].isin(["2021", "2022"])
].pivot(
    index="Experiment",
    columns="Period",
    values="Macro_F1"
)

yearly["Change_2021"] = (
    yearly["2021"] - yearly.loc["All_23", "2021"]
)

yearly["Change_2022"] = (
    yearly["2022"] - yearly.loc["All_23", "2022"]
)

yearly = yearly.reindex(my_ablation_sets.keys())

print("\n=== MALAYSIA — YEARLY ABLATION STABILITY ===")

display(
    yearly.round(4).reset_index()
)

my_tuned_ablation_df.to_csv(
    output_dir / "my_tuned_feature_ablation.csv",
    index=False
)


=== MALAYSIA — TUNED FEATURE ABLATION (2021–2022) ===


,Experiment,Features,Accuracy,Balanced_Accuracy,Macro_F1,F1_Change,Down_Recall,Flat_Recall,Up_Recall,Directional_Recall,Signal_Precision,Opposite_Error
0,All_23,23,54.6628,54.6086,0.5395,0.0000,50.2467,76.1438,37.4352,43.8409,45.7153,39.4195
1,Remove_Returns,20,44.4469,44.4673,0.4397,-0.0997,43.6223,58.5621,31.2176,37.4199,38.7267,37.3945
2,Remove_Momentum,21,54.3067,54.2690,0.5362,-0.0033,50.5285,75.5556,36.7228,43.6257,45.2113,39.8920
3,Remove_Candlestick,21,53.4609,53.4113,0.5271,-0.0124,49.1191,75.8170,35.2979,42.2085,43.8714,41.1407
4,Remove_Moving_Averages,20,53.9061,53.8762,0.5319,-0.0076,50.3876,75.4248,35.8161,43.1018,44.6951,40.2632
5,Remove_Volatility,20,53.7503,53.6958,0.5298,-0.0097,49.2600,75.8170,36.0104,42.6352,44.5826,40.1620
6,Remove_RSI,22,53.5722,53.5274,0.5283,-0.0111,49.4715,75.3595,35.7513,42.6114,44.3423,40.3982
7,Remove_MACD,20,53.7280,53.7065,0.5299,-0.0095,50.5285,75.1634,35.4275,42.9780,44.5541,40.2632
8,Remove_Volume,21,54.5738,54.5050,0.5394,-0.0001,49.6829,75.4902,38.3420,44.0124,45.7012,39.3520
9,Remove_Bollinger_Bands,20,54.1286,54.0658,0.5344,-0.0051,49.4010,75.5556,37.2409,43.3210,45.1043,39.7908



=== MALAYSIA — YEARLY ABLATION STABILITY ===


Period,Experiment,2021,2022,Change_2021,Change_2022
0,All_23,0.5361,0.5422,0.0000,0.0000
1,Remove_Returns,0.4479,0.4313,-0.0882,-0.1109
2,Remove_Momentum,0.5306,0.5411,-0.0055,-0.0011
3,Remove_Candlestick,0.5320,0.5218,-0.0041,-0.0204
4,Remove_Moving_Averages,0.5275,0.5361,-0.0086,-0.0061
5,Remove_Volatility,0.5241,0.5346,-0.0121,-0.0076
6,Remove_RSI,0.5251,0.5306,-0.0110,-0.0116
7,Remove_MACD,0.5304,0.5293,-0.0057,-0.0129
8,Remove_Volume,0.5325,0.5457,-0.0036,0.0034
9,Remove_Bollinger_Bands,0.5333,0.5346,-0.0028,-0.0076


**Prepare 2023–2024 Development Data**

In [36]:

# Use historical observations through 2024 only.
development_source = df.loc[
    pd.to_datetime(df["Date"]) < "2025-01-01"
].copy()

development_source["Date"] = pd.to_datetime(
    development_source["Date"]
)

development_source = development_source.sort_values(
    ["Ticker", "Segment_ID", "Date"]
).reset_index(drop=True)

# Recalculate future closes within each continuous stock segment,
# without using observations from 2025–2026.
segments = development_source.groupby(
    ["Ticker", "Segment_ID"], sort=False
)

future_cols = []

for h in range(1, 11):
    col = f"future_close_{h}"
    development_source[col] = segments["Close"].shift(-h)
    future_cols.append(col)

development_source["future_date_10"] = (
    segments["Date"].shift(-10)
)

# Reuse Notebook 06's common-sample eligibility requirements.
vol_cols = [
    f"volatility_{v}" for v in range(2, 11)
]

valid_volatility = (
    np.isfinite(
        development_source[vol_cols].to_numpy(dtype=float)
    ).all(axis=1)
    & development_source[vol_cols].gt(0).all(axis=1)
)

valid_future = (
    development_source[future_cols].notna().all(axis=1)
)

valid_features = (
    development_source[feature_columns].notna().all(axis=1)
)

development_mask = (
    development_source["Date"].between(
        "2023-01-01", "2024-12-31"
    )
    & (development_source["future_date_10"] < "2025-01-01")
    & valid_volatility
    & valid_future
    & valid_features
)

# Only the two selected Triple Barrier configurations.
selected_label_specs = [
    ("Malaysia", "C_Market_Specific"),
    ("Singapore", "B_Common")
]

development_label_frames = []
label_verification = []

for country, approach in selected_label_specs:

    v, h, m = candidate_configs[approach][country]

    market_source = development_source.loc[
        development_source["Country"] == country
    ].copy()

    # Reuse the existing Notebook 06 labelling function.
    generated = generate_triple_barrier_labels(
        market_source,
        vol_window=v,
        horizon=h,
        multiplier=m
    )

    generated["Date"] = pd.to_datetime(generated["Date"])
    generated["Event_End_Date"] = pd.to_datetime(
        generated["Event_End_Date"]
    )

    # Verify regenerated labels against our original 2015–2022 data.
    previous = candidate_data.loc[
        (candidate_data["Country"] == country)
        & (candidate_data["Approach"] == approach),
        ["Ticker", "Date", "Event_End_Date", "Label"]
    ].copy()

    previous["Date"] = pd.to_datetime(previous["Date"])
    previous["Event_End_Date"] = pd.to_datetime(
        previous["Event_End_Date"]
    )

    comparison = previous.merge(
        generated[
            ["Ticker", "Date", "Event_End_Date", "Label"]
        ],
        on=["Ticker", "Date"],
        how="left",
        suffixes=("_Previous", "_Regenerated"),
        validate="one_to_one"
    )

    matching = (
        (
            comparison["Label_Previous"]
            == comparison["Label_Regenerated"]
        )
        & (
            comparison["Event_End_Date_Previous"]
            == comparison["Event_End_Date_Regenerated"]
        )
    )

    assert matching.all(), (
        f"{country}: historical label mismatch detected."
    )

    label_verification.append({
        "Country": country,
        "Configuration": f"V{v}_H{h}_M{m}",
        "Previous_Rows": len(comparison),
        "Matching_Rows": int(matching.sum()),
        "Agreement_%": matching.mean() * 100
    })

    # Select eligible development observations.
    market_development = development_source.loc[
        development_mask
        & (development_source["Country"] == country),
        ["Ticker", "Country", "Date", "future_date_10"]
        + feature_columns
    ].copy()

    market_development = market_development.merge(
        generated[
            ["Ticker", "Date", "Event_End_Date", "Label"]
        ],
        on=["Ticker", "Date"],
        how="inner",
        validate="one_to_one"
    )

    assert (
        market_development["Event_End_Date"] < "2025-01-01"
    ).all()

    market_development["Approach"] = approach
    market_development["V"] = v
    market_development["H"] = h
    market_development["M"] = m
    market_development["Split"] = "Development"

    development_label_frames.append(market_development)

development_selected_data = pd.concat(
    development_label_frames,
    ignore_index=True
)

print("=== HISTORICAL LABEL VERIFICATION ===")
display(pd.DataFrame(label_verification).round(4))

# Summarise the three classes for each market and year.
development_summary = (
    development_selected_data
    .assign(Year=lambda x: x["Date"].dt.year)
    .groupby(["Country", "Approach", "Year"])
    .agg(
        Rows=("Label", "size"),
        Banks=("Ticker", "nunique"),
        Down=("Label", lambda x: int((x == -1).sum())),
        Flat=("Label", lambda x: int((x == 0).sum())),
        Up=("Label", lambda x: int((x == 1).sum()))
    )
    .reset_index()
)

for label in ["Down", "Flat", "Up"]:
    development_summary[f"{label}_Pct"] = (
        development_summary[label]
        / development_summary["Rows"] * 100
    )

print("\n=== 2023–2024 DEVELOPMENT LABEL DISTRIBUTION ===")
display(development_summary.round(2))

development_selected_data.to_csv(
    output_dir / "selected_development_labels_2023_2024.csv",
    index=False
)


=== HISTORICAL LABEL VERIFICATION ===


,Country,Configuration,Previous_Rows,Matching_Rows,Agreement_%
0,Malaysia,V2_H8_M2.5,17685,17685,100.0
1,Singapore,V2_H2_M1.0,5852,5852,100.0



=== 2023–2024 DEVELOPMENT LABEL DISTRIBUTION ===


,Country,Approach,Year,Rows,Banks,Down,Flat,Up,Down_Pct,Flat_Pct,Up_Pct
0,Malaysia,C_Market_Specific,2023,2335,10,756,775,804,32.38,33.19,34.43
1,Malaysia,C_Market_Specific,2024,2254,10,650,772,832,28.84,34.25,36.91
2,Singapore,B_Common,2023,746,3,269,240,237,36.06,32.17,31.77
3,Singapore,B_Common,2024,724,3,200,240,284,27.62,33.15,39.23


**Evaluate Selected Models on 2023–2024**

In [37]:

import numpy as np
import pandas as pd

from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    recall_score
)

development_model_specs = {
    "Malaysia": {
        "Approach": "C_Market_Specific",
        "Alpha": 0.5,
        "Window": "Expanding",
        "Tuned_Name": "Tuned_Trial_86"
    },
    "Singapore": {
        "Approach": "B_Common",
        "Alpha": 1.0,
        "Window": "Rolling_208w",
        "Tuned_Name": "Tuned_Trial_28"
    }
}


def calculate_development_metrics(actual, predicted):

    recalls = recall_score(
        actual, predicted,
        labels=[0, 1, 2],
        average=None,
        zero_division=0
    )

    actual_directional = np.isin(actual, [0, 2])
    predicted_directional = np.isin(predicted, [0, 2])

    correct_signals = (
        predicted_directional & (actual == predicted)
    )

    opposite_errors = (
        ((actual == 0) & (predicted == 2))
        | ((actual == 2) & (predicted == 0))
    )

    return {
        "Accuracy": accuracy_score(actual, predicted) * 100,
        "Balanced_Accuracy":
            balanced_accuracy_score(actual, predicted) * 100,
        "Macro_F1": f1_score(
            actual, predicted,
            labels=[0, 1, 2],
            average="macro",
            zero_division=0
        ),
        "Down_Recall": recalls[0] * 100,
        "Flat_Recall": recalls[1] * 100,
        "Up_Recall": recalls[2] * 100,
        "Directional_Recall":
            (recalls[0] + recalls[2]) * 50,
        "Signal_Precision": (
            correct_signals.sum()
            / predicted_directional.sum() * 100
            if predicted_directional.any() else 0.0
        ),
        "Opposite_Error": (
            opposite_errors.sum()
            / actual_directional.sum() * 100
            if actual_directional.any() else 0.0
        )
    }


development_evaluation_results = []

for country, config in development_model_specs.items():

    approach = config["Approach"]
    alpha = config["Alpha"]
    tuned_name = config["Tuned_Name"]

    # Reuse the original historical labels and newly prepared labels.
    historical = candidate_data.loc[
        (candidate_data["Country"] == country)
        & (candidate_data["Approach"] == approach)
    ].copy()

    development = development_selected_data.loc[
        (development_selected_data["Country"] == country)
        & (development_selected_data["Approach"] == approach)
    ].copy()

    historical["Date"] = pd.to_datetime(historical["Date"])
    development["Date"] = pd.to_datetime(development["Date"])

    historical["Event_End_Date"] = pd.to_datetime(
        historical["Event_End_Date"]
    )

    development["Event_End_Date"] = pd.to_datetime(
        development["Event_End_Date"]
    )

    required_columns = [
        "Ticker", "Date", "Event_End_Date", "Label"
    ] + list(feature_columns)

    all_data = pd.concat(
        [
            historical[required_columns],
            development[required_columns]
        ],
        ignore_index=True
    ).sort_values(["Date", "Ticker"])

    assert not all_data.duplicated(
        ["Ticker", "Date"]
    ).any()

    assert all_data[feature_columns].notna().all().all()

    parameter_sets = {
        "Untuned_XGBoost": dict(model_params),
        tuned_name: dict(tuning_best_params[country])
    }

    prediction_history = {
        "Majority_Baseline": {"Actual": [], "Predicted": []},
        **{
            name: {"Actual": [], "Predicted": []}
            for name in parameter_sets
        }
    }

    for year in [2023, 2024]:

        # Evaluate on the entire eligible calendar-year sample.
        test = development.loc[
            development["Date"].dt.year == year
        ].sort_values(["Date", "Ticker"])

        cutoff = test["Date"].min()

        # Exclude training labels that had not matured by the cutoff.
        train = all_data.loc[
            (all_data["Date"] < cutoff)
            & (all_data["Event_End_Date"] < cutoff)
        ].copy()

        # Apply the selected training window for each market.
        if config["Window"] == "Rolling_208w":
            train = train.loc[
                train["Date"] >= (
                    cutoff - pd.Timedelta(weeks=208)
                )
            ]

        train = train.sort_values(["Date", "Ticker"])

        assert len(train) > 0 and len(test) > 0
        assert train["Event_End_Date"].max() < cutoff

        X_train = train[feature_columns]
        X_test = test[feature_columns]

        y_train = train["Label"].to_numpy(dtype=int) + 1
        y_test = test["Label"].to_numpy(dtype=int) + 1

        counts = np.bincount(y_train, minlength=3)
        assert (counts > 0).all()

        # Class weights use the current training window only.
        balanced_weights = len(y_train) / (3.0 * counts)
        sample_weights = balanced_weights[y_train] ** alpha
        sample_weights /= sample_weights.mean()

        # Majority-class baseline is determined from training labels.
        majority_predictions = np.full(
            len(y_test),
            np.argmax(counts),
            dtype=int
        )

        yearly_predictions = {
            "Majority_Baseline": majority_predictions
        }

        for model_name, params in parameter_sets.items():

            model = XGBClassifier(**params)

            model.fit(
                X_train,
                y_train,
                sample_weight=sample_weights,
                verbose=False
            )

            yearly_predictions[model_name] = (
                model.predict(X_test).astype(int)
            )

        for model_name, predictions in yearly_predictions.items():

            metrics = calculate_development_metrics(
                y_test, predictions
            )

            development_evaluation_results.append({
                "Country": country,
                "Period": str(year),
                "Model": model_name,
                "Train_Rows": len(train),
                "Test_Rows": len(test),
                **metrics
            })

            prediction_history[model_name]["Actual"].append(
                y_test
            )

            prediction_history[model_name]["Predicted"].append(
                predictions
            )

    # Calculate pooled results from the 2023 and 2024 predictions.
    for model_name, history in prediction_history.items():

        actual = np.concatenate(history["Actual"])
        predicted = np.concatenate(history["Predicted"])

        metrics = calculate_development_metrics(
            actual, predicted
        )

        development_evaluation_results.append({
            "Country": country,
            "Period": "2023-2024",
            "Model": model_name,
            "Train_Rows": np.nan,
            "Test_Rows": len(actual),
            **metrics
        })


development_evaluation_df = pd.DataFrame(
    development_evaluation_results
)

performance_columns = [
    "Period",
    "Model",
    "Test_Rows",
    "Accuracy",
    "Balanced_Accuracy",
    "Macro_F1",
    "Down_Recall",
    "Flat_Recall",
    "Up_Recall",
    "Directional_Recall",
    "Signal_Precision",
    "Opposite_Error"
]

for country in ["Malaysia", "Singapore"]:

    print(
        f"\n=== {country.upper()} — "
        "2023–2024 DEVELOPMENT PERFORMANCE ==="
    )

    country_results = development_evaluation_df.loc[
        development_evaluation_df["Country"] == country,
        performance_columns
    ]

    display(
        country_results.round(4).reset_index(drop=True)
    )


development_evaluation_df.to_csv(
    output_dir / "selected_models_development_2023_2024.csv",
    index=False
)



=== MALAYSIA — 2023–2024 DEVELOPMENT PERFORMANCE ===


,Period,Model,Test_Rows,Accuracy,Balanced_Accuracy,Macro_F1,Down_Recall,Flat_Recall,Up_Recall,Directional_Recall,Signal_Precision,Opposite_Error
0,2023,Majority_Baseline,2335,32.3769,33.3333,0.1631,100.0000,0.0000,0.0000,50.0000,32.3769,51.5385
1,2023,Untuned_XGBoost,2335,50.4497,50.8293,0.5030,63.7566,60.0000,28.7313,46.2440,41.6472,44.1667
2,2023,Tuned_Trial_86,2335,50.7923,50.9764,0.5091,50.6614,66.3226,35.9453,43.3033,42.0000,42.7564
3,2024,Majority_Baseline,2254,28.8376,33.3333,0.1492,100.0000,0.0000,0.0000,50.0000,28.8376,56.1404
4,2024,Untuned_XGBoost,2254,51.8634,52.8459,0.5154,59.8462,68.5233,30.1683,45.0072,42.0776,43.0499
5,2024,Tuned_Trial_86,2254,53.1943,53.5591,0.5253,50.9231,75.2591,34.4952,42.7091,43.3684,41.5655
6,2023-2024,Majority_Baseline,4589,30.6385,33.3333,0.1564,100.0000,0.0000,0.0000,50.0000,30.6385,53.7804
7,2023-2024,Untuned_XGBoost,4589,51.1440,51.8881,0.5098,61.9488,64.2534,29.4621,45.7054,41.8497,43.6226
8,2023-2024,Tuned_Trial_86,4589,51.9721,52.2574,0.5174,50.7824,70.7822,35.2078,42.9951,42.6446,42.1762



=== SINGAPORE — 2023–2024 DEVELOPMENT PERFORMANCE ===


,Period,Model,Test_Rows,Accuracy,Balanced_Accuracy,Macro_F1,Down_Recall,Flat_Recall,Up_Recall,Directional_Recall,Signal_Precision,Opposite_Error
0,2023,Majority_Baseline,746,31.7694,33.3333,0.1607,0.0000,0.0000,100.0000,50.0000,31.7694,53.1621
1,2023,Untuned_XGBoost,746,46.1126,46.1207,0.4664,44.9814,56.2500,37.1308,41.0561,38.7755,44.4664
2,2023,Tuned_Trial_28,746,46.5147,46.2952,0.4670,50.5576,55.4167,32.9114,41.7345,39.4834,43.6759
3,2024,Majority_Baseline,724,39.2265,33.3333,0.1878,0.0000,0.0000,100.0000,50.0000,39.2265,41.3223
4,2024,Untuned_XGBoost,724,46.4088,47.5239,0.4688,53.0000,55.4167,34.1549,43.5775,39.1892,42.9752
5,2024,Tuned_Trial_28,724,47.2376,48.8654,0.4749,57.5000,59.1667,29.9296,43.7148,39.3701,43.3884
6,2023-2024,Majority_Baseline,1470,35.4422,33.3333,0.1745,0.0000,0.0000,100.0000,50.0000,35.4422,47.3737
7,2023-2024,Untuned_XGBoost,1470,46.2585,46.5809,0.4678,48.4009,55.8333,35.5086,41.9547,38.9782,43.7374
8,2023-2024,Tuned_Trial_28,1470,46.8707,47.3653,0.4713,53.5181,57.2917,31.2860,42.4021,39.4286,43.5354


**Bank-level development stability**

In [38]:

bank_development_rows = []
bank_prediction_records = []

for country, config in development_model_specs.items():

    approach = config["Approach"]
    alpha = config["Alpha"]

    historical = candidate_data.loc[
        (candidate_data["Country"] == country)
        & (candidate_data["Approach"] == approach)
    ].copy()

    development = development_selected_data.loc[
        (development_selected_data["Country"] == country)
        & (development_selected_data["Approach"] == approach)
    ].copy()

    required = [
        "Ticker", "Date", "Event_End_Date", "Label"
    ] + list(feature_columns)

    history = pd.concat(
        [historical[required], development[required]],
        ignore_index=True
    )

    history["Date"] = pd.to_datetime(history["Date"])
    history["Event_End_Date"] = pd.to_datetime(
        history["Event_End_Date"]
    )

    parameter_sets = {
        "Untuned": dict(model_params),
        "Tuned": dict(tuning_best_params[country])
    }

    for year in [2023, 2024]:

        test = development.loc[
            development["Date"].dt.year == year
        ].sort_values(["Date", "Ticker"])

        cutoff = pd.to_datetime(test["Date"]).min()

        # Only labels available before the yearly prediction cutoff.
        train = history.loc[
            (history["Date"] < cutoff)
            & (history["Event_End_Date"] < cutoff)
        ].sort_values(["Date", "Ticker"])

        if config["Window"] == "Rolling_208w":
            train = train.loc[
                train["Date"] >= (
                    cutoff - pd.Timedelta(weeks=208)
                )
            ]

        y_train = train["Label"].to_numpy(dtype=int) + 1
        y_test = test["Label"].to_numpy(dtype=int) + 1

        counts = np.bincount(y_train, minlength=3)
        assert (counts > 0).all()

        class_weights = len(y_train) / (3.0 * counts)
        sample_weights = class_weights[y_train] ** alpha
        sample_weights /= sample_weights.mean()

        tickers = test["Ticker"].to_numpy()

        for model_name, params in parameter_sets.items():

            model = XGBClassifier(**params)

            model.fit(
                train[feature_columns],
                y_train,
                sample_weight=sample_weights,
                verbose=False
            )

            predictions = model.predict(
                test[feature_columns]
            ).astype(int)

            for ticker in np.unique(tickers):

                mask = tickers == ticker

                actual_bank = y_test[mask]
                predicted_bank = predictions[mask]

                metrics = calculate_development_metrics(
                    actual_bank, predicted_bank
                )

                bank_development_rows.append({
                    "Country": country,
                    "Bank": ticker,
                    "Year": str(year),
                    "Model": model_name,
                    "Rows": int(mask.sum()),
                    **metrics
                })

                bank_prediction_records.append(
                    pd.DataFrame({
                        "Country": country,
                        "Bank": ticker,
                        "Year": year,
                        "Model": model_name,
                        "Actual": actual_bank,
                        "Predicted": predicted_bank
                    })
                )

# Calculate pooled 2023–2024 performance for each bank.
bank_prediction_df = pd.concat(
    bank_prediction_records,
    ignore_index=True
)

for (country, bank, model_name), group in (
    bank_prediction_df.groupby(["Country", "Bank", "Model"])
):

    metrics = calculate_development_metrics(
        group["Actual"].to_numpy(),
        group["Predicted"].to_numpy()
    )

    bank_development_rows.append({
        "Country": country,
        "Bank": bank,
        "Year": "2023-2024",
        "Model": model_name,
        "Rows": len(group),
        **metrics
    })

bank_development_df = pd.DataFrame(bank_development_rows)

metric_columns = [
    "Bank", "Model", "Rows",
    "Accuracy", "Balanced_Accuracy", "Macro_F1",
    "Down_Recall", "Flat_Recall", "Up_Recall",
    "Directional_Recall", "Signal_Precision",
    "Opposite_Error"
]

for country in ["Malaysia", "Singapore"]:

    pooled = bank_development_df.loc[
        (bank_development_df["Country"] == country)
        & (bank_development_df["Year"] == "2023-2024")
    ].sort_values(["Bank", "Model"])

    print(
        f"\n=== {country.upper()} — "
        "BANK-LEVEL PERFORMANCE (2023–2024) ==="
    )

    display(
        pooled[metric_columns].round(4).reset_index(drop=True)
    )

    # Positive changes mean the tuned model improved Macro F1.
    yearly = bank_development_df.loc[
        (bank_development_df["Country"] == country)
    ].pivot(
        index="Bank",
        columns=["Year", "Model"],
        values="Macro_F1"
    )

    changes = pd.DataFrame({
        "Bank": yearly.index,
        "F1_Change_2023": (
            yearly[("2023", "Tuned")]
            - yearly[("2023", "Untuned")]
        ).to_numpy(),
        "F1_Change_2024": (
            yearly[("2024", "Tuned")]
            - yearly[("2024", "Untuned")]
        ).to_numpy(),
        "F1_Change_Overall": (
            yearly[("2023-2024", "Tuned")]
            - yearly[("2023-2024", "Untuned")]
        ).to_numpy()
    })

    print(
        f"\n=== {country.upper()} — "
        "BANK-LEVEL TUNING IMPROVEMENT ==="
    )

    display(changes.round(4).reset_index(drop=True))

bank_development_df.to_csv(
    output_dir / "selected_bank_development_performance.csv",
    index=False
)

bank_prediction_df.to_csv(
    output_dir / "selected_bank_development_predictions.csv",
    index=False
)



=== MALAYSIA — BANK-LEVEL PERFORMANCE (2023–2024) ===


,Bank,Model,Rows,Accuracy,Balanced_Accuracy,Macro_F1,Down_Recall,Flat_Recall,Up_Recall,Directional_Recall,Signal_Precision,Opposite_Error
0,1015.KL,Tuned,469,53.0917,52.7882,0.5204,47.1831,77.4390,33.7423,40.4627,41.6382,43.9344
1,1015.KL,Untuned,469,56.7164,56.7220,0.5603,57.7465,75.6098,36.8098,47.2781,46.7105,40.0000
2,1023.KL,Tuned,474,51.4768,53.4077,0.5226,45.6522,73.6842,40.8867,43.2694,44.9231,42.2287
3,1023.KL,Untuned,474,51.8987,54.4823,0.5334,55.0725,71.4286,36.9458,46.0091,44.2815,44.5748
4,1066.KL,Tuned,468,47.6496,47.6996,0.4726,51.6393,63.4831,27.9762,39.8078,33.6391,52.4138
5,1066.KL,Untuned,468,41.0256,43.4988,0.4060,68.8525,44.3820,17.2619,43.0572,30.2949,55.5172
6,1155.KL,Tuned,472,47.8814,48.3848,0.4928,44.1718,63.9456,37.0370,40.6044,37.8223,50.4615
7,1155.KL,Untuned,472,48.0932,48.0273,0.4773,73.0061,47.6190,23.4568,48.2315,40.0510,48.6154
8,1171.KL,Tuned,451,49.6674,49.4573,0.4861,44.3609,70.8861,33.1250,38.7430,41.0256,39.2491
9,1171.KL,Untuned,451,50.1109,50.1648,0.4913,49.6241,69.6203,31.2500,40.4370,41.7266,38.9078



=== MALAYSIA — BANK-LEVEL TUNING IMPROVEMENT ===


,Bank,F1_Change_2023,F1_Change_2024,F1_Change_Overall
0,1015.KL,-0.0453,-0.0337,-0.0398
1,1023.KL,-0.0279,0.0111,-0.0108
2,1066.KL,0.0765,0.0563,0.0666
3,1155.KL,0.0390,0.0003,0.0156
4,1171.KL,-0.0046,-0.0061,-0.0052
5,1295.KL,0.0279,-0.0051,0.0133
6,2488.KL,0.0284,0.0427,0.0324
7,5185.KL,-0.0123,0.0467,0.0196
8,5258.KL,0.0000,-0.0107,-0.0040
9,5819.KL,0.0356,-0.0072,0.0118



=== SINGAPORE — BANK-LEVEL PERFORMANCE (2023–2024) ===


,Bank,Model,Rows,Accuracy,Balanced_Accuracy,Macro_F1,Down_Recall,Flat_Recall,Up_Recall,Directional_Recall,Signal_Precision,Opposite_Error
0,D05.SI,Tuned,491,45.6212,45.5172,0.4560,46.8750,56.5476,33.1288,40.0019,38.1657,42.1053
1,D05.SI,Untuned,491,46.4358,46.3397,0.4678,45.6250,55.3571,38.0368,41.8309,38.5714,43.3437
2,O39.SI,Tuned,488,48.3607,49.6762,0.4857,61.3793,59.6273,28.0220,44.7006,39.6601,45.2599
3,O39.SI,Untuned,488,46.1066,46.8421,0.4669,50.3448,57.7640,32.4176,41.3812,37.6068,46.1774
4,U11.SI,Tuned,491,46.6395,47.2108,0.4717,53.0488,55.6291,32.9545,43.0017,40.3900,43.2353
5,U11.SI,Untuned,491,46.2322,46.6862,0.4680,49.3902,54.3046,36.3636,42.8769,40.7303,41.7647



=== SINGAPORE — BANK-LEVEL TUNING IMPROVEMENT ===


,Bank,F1_Change_2023,F1_Change_2024,F1_Change_Overall
0,D05.SI,-0.0253,0.0008,-0.0117
1,O39.SI,0.0160,0.0174,0.0187
2,U11.SI,0.0083,-0.0012,0.0037
